# Master Donor Feature Schema for DonorsChoose

## Overview

This notebook implements a comprehensive, reusable donor feature engineering pipeline that:

1. **Re-uses existing feature engineering work** while consolidating near-duplicate concepts
2. **Separates features by time scope**:
   - `STATIC`: Does not depend on reference time T (or treated as fixed once per donor)
   - `AS_OF_T`: Uses cumulative data up to time T
   - `WINDOWED`: Uses specific lookback windows relative to T
   - `LABEL`: Natural target variables for various use cases

3. **Pulls from 5 main data sources**:
   - Donor Project Records (gifts/donations)
   - Email Events (12-month summary)
   - Site Events (FY25-26 activity)
   - Monthly Donation Program (all-time)
   - Share Events (all-time)
   - Plus: ZIP-level ACS demographics

## Feature Categories

The schema includes approximately 150+ features across these domains:

1. **Identity & Demographics** (ZIP-based ACS)
2. **Lifetime Giving Behavior** (tenure, amounts, patterns)
3. **Windowed Giving** (3m, 12m, 36m activity + velocity)
4. **Channel/Payment Mix** (DAF, green, gift cards, etc.)
5. **Monthly Program Dynamics** (subscription behavior)
6. **Teacher/School/Content Preferences** (loyalty vs diversification)
7. **Seasonality** (back-to-school, year-end, etc.)
8. **Email Engagement** (opens, clicks, velocity)
9. **Site Behavior** (sessions, pages, devices)
10. **Share Activity** (social sharing patterns)
11. **Project Outcomes** (fully-funded rates, matching)
12. **Labels** (optional, for various prediction tasks)

## Key Design Principles

- **No duplicate concepts**: Single distance metric, single velocity calculation pattern
- **Consistent windowing**: 3-month (short), 12-month (mid), 36-month (long)
- **Defensive coding**: Small epsilon values prevent division by zero
- **Separation of concerns**: Feature building ≠ imputation/encoding
- **Time-relative**: All features parameterized by reference time T


## v1.3 changelog (2026-09-30)

**Fixed**
- **Monthly charge date.** The monthly extract has `monthly_subscription_payment_date`, not `charge_date`, so every `charge_date` branch silently did nothing. `build_features` now aliases it. This restores `monthly_amount_12m`, `months_since_last_monthly_charge` and `pct_amount_monthly_12m` (previously 0 for everyone), and the monthly-dependent labels: monthly revenue, grand revenue, midlevel/tiptop crossing, monthly amount 6m, upgrade/downgrade.
- **Monthly leakage.** `md_pre_T` was filtered on join date only, so lifetime/median monthly amounts included payments after T. Payment-based aggregates now use charges before T; longest streak is recomputed from pre-T charges (the extract's streak is a current value).
- **Site sessions.** `df_site` is one row per donor-day, so row counts were days, not sessions. `avg_sessions_per_active_day_3m` (was ≈1 for everyone) now uses `sessions_day`; `avg_session_duration_min_3m` (was missing) uses `duration_sum_min`; cart visits are per session.
- **`latest_referral_channel`.** Source whitelist matched the SQL buckets (was `{'dc','google'}`, which don't exist in the extract).
- **`pct_amount_to_top_*`.** Denominator excludes rows with no teacher/school/category/grade.
- **Account credit.** All credit features and `label_will_use_account_credit_next_90d` now come from `AccountCredits.csv` (loaded in the run cell). The old `ever_used_account_credit` / `current_account_credit_balance` (built from `df_dpr.account_credit_balance`, which is today's balance on every row) are removed, as is the label's `account_credit_amount` branch (that column never existed, so the label was all 0).

**Added**
- 2.13 behavioral expansion (~63 features), 2.14 credit ledger from `AccountCredits.csv` (9 features + label).
- `label_will_make_daf_vehicle_gift_next_365d` / `label_daf_vehicle_revenue_next_365d` when the extract has `is_daf_vehicle`.
- `build_features(..., df_credit=None, apply_loyalist_rule_at_T=False)`; the run cell passes the ledger and sets the rule on.
- `SNAPSHOT_FEATURES` and `SUPERSEDED_FEATURES` lists for the modeling pipeline.

**v1.3.1 fixes:** input checks no longer assume parsed dates (they run on the raw CSVs); occasion keys use a resolution-independent day count; payment-type mix is gift-level (`pct_gifts_paytype_*`, since 4.71% of checkouts mix payment types); `AccountCredits.csv` loads with the other extracts and its headers are normalized; site features tolerate a missing `sessions_day`. Pre-existing bugs fixed: `pct_count_green_*` was 0 for everyone (`is_green_payment` is 't'/'f'); identity flags were NaN for monthly-only donors; `share_month_coverage_ratio` mixed a 12m numerator with a lifetime denominator; `label_num_sessions_next_90d` counted days; the next-campaign open/click labels are now built from monthly Campaign rows; labels with no source data in the extract (unsubscribe, login) are no longer created, instead of being imputed to all-0.

**v1.3.2 credit features:** removed `credit_amount_24m` and `credit_share_of_giving_capacity_12m` (each > 0.95 correlated with an existing feature). Added `credit_amount_lifetime`, `max_single_credit_lifetime`, `n_credits_250plus_lifetime`, `largest_credit_share_of_lifetime_credits`, `days_since_largest_credit`. Credit features now: 9 (was 6).

**v1.3.3:** (1) `label_will_give_2plus_times_next_365d` counts checkout occasions, not payment rows; (2) `_occasion_key` cascades per row cart_id → donation_n → donor-day with non-colliding key ranges; (3) `latest_*` features come from one actual row (was column-wise `last()`); (4) `label_will_make_major_gift_next_24m` is now built (single payment ≥ $10k in 24m; NaN when the window isn't fully observed, e.g. OOT); (5) 8 current-state features are dropped from the output (`SNAPSHOT_DROPPED_FEATURES`); `is_major_gift_donor` and `is_marketing_subscribed` stay for cohort/opt-out filters and must be excluded from model inputs downstream.

**v1.3.4:** revenue labels are indexed on project and monthly donors (monthly-only donors previously got 0 for `label_monthly_revenue_next_365d`, `label_grand_revenue_next_365d` and the midlevel/tiptop crossing labels); `label_will_be_midlevel_grand_next_12m` now uses grand revenue (project + monthly) instead of project dollars only.

**v1.3.5 (modeling contract):** each frame is written with a `<stem>.manifest.json` (source history start/end, per-feature group/sources/lookback/snapshot/sensitive/model_input, per-label window/sources/maturity) and a `<stem>.features.csv` review copy; the run cell also builds a `score` frame at T = DATA_ASOF. Open-ended label windows are now fixed so maturity is deterministic: `label_time_to_next_gift_days`, `label_next_gift_category`, `label_next_gift_grade_band` and `label_optional_donation_rate_on_next_gift` use the first gift within 365 days; the next-campaign open/click labels use the first campaign month within 90 days.

**v1.3.6:** missing source information is never treated as observed. A dynamic source with no data gets `history_start` / `data_end` = null; any label reading it has `observed_through` = null, `mature` = false and lists it in `missing_sources`. Static sources are flagged `static: true`.

**v1.3.7:** the run cell keeps your naming (`train_features_{FRAME_TAG}` / `oot_features_{FRAME_TAG}`, FRAME_TAG = `t<T>_oot<T+O>-<T+O+H>`) and writes a manifest beside every frame. A separate `score_features_t<DATA_ASOF>` frame is built only when T + O is earlier than DATA_ASOF; with T = DATA_ASOF − 365 the oot frame already is the current frame. The cell prints the modeling notebook's VALIDATE_FRAMES / SCORE_FRAMES for the files it wrote.

**Expect after rerun:** monthly/grand revenue labels change, the training population shrinks under the as-of-T loyalist rule, and the frame gains ~75 columns.

**Not changed (decisions/recommendations):** `daf_payment` definition, snapshot features left in the frame, velocity epsilon, email velocity windows. See the consolidated review document.


## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
from pyproj import Geod
from typing import Optional, Tuple
import warnings
import gc
import psutil
from pathlib import Path
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. Helper Functions

In [2]:
# Label configuration constants
# ---------------------------------------------------------------------

MIDLEVEL_THRESHOLD = 1000           # annual midlevel threshold for "grand" revenue
TIPTOP_THRESHOLD = 10000            # threshold for "major gift" classification
LARGE_SINGLE_GIFT_THRESHOLD = 500   # large single gift threshold (e.g., $500)
TIME_TO_NEXT_GIFT_CENSOR_DAYS = 366 # censoring value for time-to-next-gift labels


# ---------------------------------------------------------------------
# Core validation and utility helpers
# ---------------------------------------------------------------------

def _validate_dpr_for_labels(df_dpr: pd.DataFrame) -> None:
    """
    Ensure df_dpr has the minimum required columns for label construction.
    Raises a ValueError if core columns are missing.
    """
    required = {"donor_id", "payment_date", "payment_amount"}
    missing = required - set(df_dpr.columns)
    if missing:
        raise ValueError(
            f"df_dpr is missing required columns for label construction: {missing}"
        )


def _future_window(df, date_col: str, T: pd.Timestamp, horizon_days: int) -> pd.DataFrame:
    """
    Generic helper: slice df to [T, T + horizon_days).
    Returns an empty DataFrame if df is None or empty.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=[date_col])
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    end = T + pd.Timedelta(days=horizon_days)
    mask = (df[date_col] >= T) & (df[date_col] < end)
    return df.loc[mask]


def _any_by_donor(df, donor_col: str = "donor_id") -> pd.Series:
    """
    Return 1/0 per donor for "any rows in df".
    Empty input returns an empty Series.
    """
    if df is None or len(df) == 0:
        return pd.Series([], dtype="float64", name=donor_col)
    return df.groupby(donor_col).size().gt(0).astype(float)


def _count_by_donor(df, donor_col: str = "donor_id") -> pd.Series:
    """
    Return count of rows per donor.
    """
    if df is None or len(df) == 0:
        return pd.Series([], dtype="float64", name=donor_col)
    return df.groupby(donor_col).size().astype(float)


def _sum_by_donor(df, value_col: str, donor_col: str = "donor_id") -> pd.Series:
    """
    Sum of value_col per donor.
    """
    if df is None or len(df) == 0:
        return pd.Series([], dtype="float64", name=donor_col)
    return df.groupby(donor_col)[value_col].sum().astype(float)


def _max_by_donor(df, value_col: str, donor_col: str = "donor_id") -> pd.Series:
    """
    Max of value_col per donor.
    """
    if df is None or len(df) == 0:
        return pd.Series([], dtype="float64", name=donor_col)
    return df.groupby(donor_col)[value_col].max().astype(float)


def haversine_miles(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance between two points in miles using Haversine formula.
    
    All arguments can be scalars or pandas Series (for vectorized computation).
    
    Parameters
    ----------
    lat1, lon1 : float or pd.Series
        Latitude and longitude of first point(s) in decimal degrees
    lat2, lon2 : float or pd.Series
        Latitude and longitude of second point(s) in decimal degrees
        
    Returns
    -------
    distance : float or pd.Series
        Great-circle distance in miles
    """
    # Earth's radius in miles
    R = 3958.8
    
    # Convert all coordinates from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Calculate differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine formula
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

def entropy_vectorized(df, group_col, cat_col):
    """
    Calculate Shannon entropy in a fully vectorized way.
        
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the data
    group_col : str
        Column to group by (e.g., 'donor_id')
    cat_col : str
        Categorical column to calculate entropy for (e.g., 'teacher_id')
    
    Returns
    -------
    entropy : pd.Series
        Entropy value for each group, indexed by group_col
    """
    if df.empty:
        return pd.Series(dtype=float, name='entropy')
    
    # Count occurrences of each category within each group
    counts = df.groupby([group_col, cat_col]).size()
    
    # Calculate probabilities (normalize within each group)
    probs = counts.groupby(level=0).transform(lambda x: x / x.sum())
    
    # Calculate entropy: -sum(p * log(p)) for each group
    entropy = -(probs * np.log(probs + 1e-9)).groupby(level=0).sum()
    
    return entropy


def pct_amount(df, flag_col, amount_col='payment_amount'):
    """
    Calculate the fraction of total donation amount coming from flagged rows.
    
    Example: If flag_col='daf_payment', returns what % of each donor's
    total giving came through DAF donations.
    
    Parameters
    ----------
    df : pd.DataFrame
        Donation records with donor_id column
    flag_col : str
        Name of binary indicator column (1 = flagged, 0 = not flagged)
    amount_col : str, default='payment_amount'
        Name of amount column to aggregate
        
    Returns
    -------
    pct : pd.Series
        Fraction of amount from flagged rows, indexed by donor_id
    """

    if df.empty:
        return pd.Series(dtype=float)
    
    # Filter THEN group (vectorized, no lambda)
    flagged_df = df[df[flag_col] == 1]
    amt_flag = flagged_df.groupby('donor_id')[amount_col].sum()
    amt_total = df.groupby('donor_id')[amount_col].sum()
    amt_flag = amt_flag.reindex(amt_total.index, fill_value=0)
    
    # Return fraction (with small epsilon to avoid division by zero)
    return amt_flag / (amt_total + 1e-6)


def pct_count(df, flag_col):
    """
    Calculate the fraction of total gift count coming from flagged rows.
    
    Example: If flag_col='gift_card_purchase', returns what % of each donor's
    gifts were gift card purchases.
    
    Parameters
    ----------
    df : pd.DataFrame
        Donation records with donor_id column
    flag_col : str
        Name of binary indicator column
        
    Returns
    -------
    pct : pd.Series
        Fraction of gifts that are flagged, indexed by donor_id
    """
    if df.empty:
        return pd.Series(dtype=float)
        
    # Filter THEN group (vectorized)
    flagged_df = df[df[flag_col] == 1]
    cnt_flag = flagged_df.groupby('donor_id').size()
    cnt_total = df.groupby('donor_id').size()
    cnt_flag = cnt_flag.reindex(cnt_total.index, fill_value=0)
    
    return cnt_flag / (cnt_total + 1e-6)

## 2. Feature Group Functions

### 2.1 Identity & Demographics

In [3]:
def _identity_and_zip_features(dpr_pre_T, df_zip_acs, md_pre_T=None):
    """
    Build identity features and ZIP-level demographics.
    
    Parameters:
    -----------
    dpr_pre_T : pd.DataFrame
        Donation records before T
    df_zip_acs : pd.DataFrame
        ZIP-level ACS demographics
    md_pre_T : pd.DataFrame, optional
        Monthly donation data (for ZIP fallback for monthly-only donors)
    """
    # Create base index from all possible donors
    all_donor_ids = dpr_pre_T['donor_id'].unique()
    if md_pre_T is not None:
        all_donor_ids = pd.Index(
            pd.unique(
                pd.concat([
                    pd.Series(dpr_pre_T['donor_id'].unique()),
                    pd.Series(md_pre_T['donor_id'].unique())
                ])
            )
        )
    
    if dpr_pre_T.empty and (md_pre_T is None or md_pre_T.empty):
        return pd.DataFrame(index=pd.Index([], name='donor_id'))

    # Get ZIP from PROJECT donations (if available)
    if not dpr_pre_T.empty:
        donor_zip5_from_dpr = (
            dpr_pre_T
            .sort_values('payment_date')
            .groupby('donor_id')['donor_zip']
            .first()
        )
    else:
        donor_zip5_from_dpr = pd.Series(dtype=object, name='donor_zip')
    
    # Get ZIP from MONTHLY data as fallback
    if md_pre_T is not None and not md_pre_T.empty and 'donor_zip' in md_pre_T.columns:
        donor_zip5_from_monthly = (
            md_pre_T
            .sort_values('monthly_subscription_joined_date')
            .groupby('donor_id')['donor_zip']
            .first()
        )
        # Combine: use dpr ZIP if available, otherwise monthly ZIP
        donor_zip5_raw = donor_zip5_from_dpr.combine_first(donor_zip5_from_monthly)
    else:
        donor_zip5_raw = donor_zip5_from_dpr

    # Binary status flags (from dpr_pre_T)
    if not dpr_pre_T.empty:
        is_teacher = dpr_pre_T.groupby('donor_id')['is_teacher'].max()
        is_teacher_referred = dpr_pre_T.groupby('donor_id')['is_teacher_referred'].max()
        is_marketing_subscribed = dpr_pre_T.groupby('donor_id')['subscribed_to_marketing_emails'].max()
        is_major_gift_donor = dpr_pre_T.groupby('donor_id')['major_gift_donor'].max()
        # v1.3: account credit features moved to the AccountCredits ledger (section 2.14);
        # df_dpr.account_credit_balance was today's balance repeated on every row.
    else:
        # Create empty series if no dpr data
        is_teacher = pd.Series(dtype=float)
        is_teacher_referred = pd.Series(dtype=float)
        is_marketing_subscribed = pd.Series(dtype=float)
        is_major_gift_donor = pd.Series(dtype=float)

    # --- Donor Type Flags (always create for all donors) ---
    # Determine which donors appear in each data source
    project_donor_ids = set(dpr_pre_T['donor_id'].unique()) if not dpr_pre_T.empty else set()
    monthly_donor_ids = set(md_pre_T['donor_id'].unique()) if md_pre_T is not None and not md_pre_T.empty else set()
    
    # Create flags for all donors in our universe
    has_project_history = pd.Series([d in project_donor_ids for d in all_donor_ids], index=all_donor_ids, dtype=int)
    has_monthly_history = pd.Series([d in monthly_donor_ids for d in all_donor_ids], index=all_donor_ids, dtype=int)
    
    # Composite flags for donor segmentation
    is_monthly_only_donor = ((has_monthly_history == 1) & (has_project_history == 0)).astype(int)
    is_project_only_donor = ((has_project_history == 1) & (has_monthly_history == 0)).astype(int)
    is_hybrid_donor = ((has_project_history == 1) & (has_monthly_history == 1)).astype(int)

    # --- ZIP / ACS JOIN (dtype-safe) ------------------------------------
    donor_zip5 = pd.to_numeric(donor_zip5_raw, errors="coerce")

    df_zip_acs = df_zip_acs.copy()
    if 'ZIP5' in df_zip_acs.columns:
        df_zip_acs = df_zip_acs.set_index('ZIP5')
    df_zip_acs.index = pd.to_numeric(df_zip_acs.index, errors="coerce")

    acs = donor_zip5.to_frame('donor_zip5').join(
        df_zip_acs,
        on='donor_zip5',
        how='left'
    )

    # Assemble output
    # v1.3.1: index on every donor (project and/or monthly). v1.2 indexed on donors with a
    # ZIP from df_dpr only, so monthly-only donors got NaN for has_project_history,
    # is_monthly_only_donor, is_hybrid_donor, etc.
    out = pd.DataFrame(index=pd.Index(pd.unique(pd.Series(all_donor_ids)), name='donor_id'))
    out['donor_zip5'] = donor_zip5
    out['has_project_history'] = has_project_history
    out['has_monthly_history'] = has_monthly_history
    out['is_monthly_only_donor'] = is_monthly_only_donor
    out['is_project_only_donor'] = is_project_only_donor
    out['is_hybrid_donor'] = is_hybrid_donor
    out['is_teacher'] = is_teacher
    out['is_teacher_referred'] = is_teacher_referred
    out['is_marketing_subscribed'] = is_marketing_subscribed
    out['is_major_gift_donor'] = is_major_gift_donor

    # ACS features
    out['zip_pct_households_with_children'] = acs['pct_households_with_children']
    out['zip_pct_in_labor_force'] = acs['pct_in_labor_force']
    out['zip_unemployment_rate'] = acs['unemployment_rate']
    out['zip_pct_single_parent'] = acs['pct_single_parent']
    out['zip_pct_minority'] = acs['pct_minority']
    out['zip_avg_household_size'] = acs['avg_household_size']
    out['zip_median_age'] = acs['median_age']
    out['zip_median_home_value'] = acs['median_home_value']
    
    if 'total_population' in acs.columns:
        out['zip_log_total_population'] = np.log1p(acs['total_population'])
    elif 'log_total_population' in acs.columns:
        out['zip_log_total_population'] = acs['log_total_population']
    else:
        out['zip_log_total_population'] = np.nan
        
    return out

### 2.2 Lifetime Giving Behavior

In [4]:
def _lifetime_giving_features(dpr_pre_T, T):
    """
    Build cumulative giving features using all donations before T.
    
    CORRECTED VERSION with:
    - Tenure relative to T (not last_donation_date)
    - Gap calculations only for last 730 days
    
    Features created:
    - first_donation_date, last_donation_date: Temporal boundaries
    - tenure_days, tenure_years, tenure_bucket: How long donor has been active (relative to T)
    - lifetime_gift_count, lifetime_amount: Total volume
    - lifetime_median/max/cv_gift_amount: Distribution of gift sizes
    - mean/cv_gap_between_gifts_days: Giving rhythm/consistency (last 730 days only)
    - max_donation_sequence_number: How many gifts total
    - pct_early_gifts_in_lifetime: Concentration in first 3 gifts
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        Donor Project Records filtered to payment_date < T
    T : pd.Timestamp
        Reference time (end of training period)
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if dpr_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    dpr = dpr_pre_T.copy()
    dpr = dpr.sort_values(['donor_id', 'payment_date'])
    
    # --- Temporal boundaries ---
    first_donation_date = dpr.groupby('donor_id')['payment_date'].min()
    last_donation_date = dpr.groupby('donor_id')['payment_date'].max()
    
    # CORRECTED: Tenure relative to T (end of training period), not last donation
    tenure_days = (T - first_donation_date).dt.days
    tenure_years = tenure_days / 365.25
    
    # Bucket tenure for non-linear patterns (categorical)
    tenure_bucket_cat = pd.cut(
        tenure_years,
        bins=[0, 1, 3, 5, 100],
        labels=['<1y', '1-3y', '3-5y', '5y+'],
        right=False
    )
    
    # Map bucket to lower-bound in YEARS as float64
    tenure_bucket = tenure_bucket_cat.map({
        '<1y': 0.0,
        '1-3y': 1.0,
        '3-5y': 3.0,
        '5y+': 5.0,
    }).astype(float)
    
    # --- Volume metrics ---
    lifetime_gift_count = dpr.groupby('donor_id')['payment_amount'].size()
    lifetime_amount = dpr.groupby('donor_id')['payment_amount'].sum()
    
    # --- Distribution of gift sizes ---
    gifts_by_donor = dpr.groupby('donor_id')['payment_amount']
    med = gifts_by_donor.median()
    mx = gifts_by_donor.max()
    mean = gifts_by_donor.mean()
    std = gifts_by_donor.std()
    # Coefficient of variation: std/mean
    # High CV = erratic giving amounts, Low CV = consistent amounts
    cv = std / (mean + 1e-9)
    
    # --- Inter-gift timing patterns (LAST 730 DAYS ONLY) ---
    # CORRECTED: Only use donations in last 730 days for gap calculations
    cutoff_730 = T - pd.Timedelta(days=730)
    dpr_730 = dpr[dpr['payment_date'] >= cutoff_730].copy()
    
    if not dpr_730.empty:
        # Calculate days between consecutive gifts
        dpr_730['prev_payment_date'] = dpr_730.groupby('donor_id')['payment_date'].shift(1)
        dpr_730['gap_days'] = (dpr_730['payment_date'] - dpr_730['prev_payment_date']).dt.days
        
        gap_by_donor = dpr_730.groupby('donor_id')['gap_days']
        gap_mean = gap_by_donor.mean()
        gap_std = gap_by_donor.std()
        # CV of gaps: High = irregular giving, Low = regular/predictable
        cv_gap = gap_std / (gap_mean + 1e-9)
    else:
        # No donations in last 730 days
        gap_mean = pd.Series(dtype=float)
        cv_gap = pd.Series(dtype=float)
    
    # --- Gift sequence metrics ---
    # donation_n = sequential gift number for this donor
    max_donation_sequence_number = dpr.groupby('donor_id')['donation_n'].max()
    
    # What fraction of gifts were in the "early stage" (first 3 gifts)?
    # High concentration here might indicate acquisition success but poor retention
    dpr['is_early_gift'] = dpr['donation_n'] <= 3
    pct_early_gifts = dpr.groupby('donor_id')['is_early_gift'].mean()
    
    # --- Assemble output ---
    out = pd.DataFrame(index=lifetime_amount.index)
    
    out['first_donation_date'] = first_donation_date
    out['last_donation_date'] = last_donation_date
    out['tenure_days'] = tenure_days
    out['tenure_years'] = tenure_years
    out['tenure_bucket'] = tenure_bucket
    
    out['lifetime_gift_count'] = lifetime_gift_count
    out['lifetime_amount'] = lifetime_amount
    out['lifetime_median_gift_amount'] = med
    out['lifetime_max_gift_amount'] = mx
    out['lifetime_cv_gift_amount'] = cv
    
    out['mean_gap_between_gifts_days_last2yr'] = gap_mean
    out['cv_gap_between_gifts_days_last2yr'] = cv_gap
    
    out['max_donation_sequence_number'] = max_donation_sequence_number
    out['pct_early_gifts_in_lifetime'] = pct_early_gifts
    
    return out

### 2.3 Windowed Giving & Velocity

These features capture recent activity and trends.

In [5]:
def _windowed_giving_features(dpr_pre_T, dpr_3m, dpr_12m, dpr_36m, dpr_3to12m, dpr_12to36m, T):
    """
    Build windowed giving features and velocity metrics.
    
    Features created:
    - gift_count/amount/median_amount for 3m, 12m, 36m windows
    - days_since_last_gift, days_since_second_to_last_gift: Recency
    - amount_velocity_0to3_vs_3to12: Is giving accelerating or decelerating?
    - amount_velocity_0to12_vs_12to36: Longer-term trend
    - count_velocity_0to12_vs_12to36: Frequency trend
    
    Velocity > 1 indicates acceleration (more recent activity)
    Velocity < 1 indicates deceleration (declining activity)
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        All donations before T
    dpr_3m, dpr_12m, dpr_36m : pd.DataFrame
        Donations in respective windows
    dpr_3to12m, dpr_12to36m : pd.DataFrame
        Intermediate periods for velocity calculations
    T : pd.Timestamp
        Reference time
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    out_index = dpr_pre_T['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # --- 3-month window (recent/short-term) ---
    if not dpr_3m.empty:
        by_3m = dpr_3m.groupby('donor_id')['payment_amount']
        out['gift_count_3m'] = by_3m.size()
        out['gift_amount_3m'] = by_3m.sum()
        out['median_gift_amount_3m'] = by_3m.median()

        # Split by green vs non-green payments
        if 'is_green_payment' in dpr_3m.columns:
            # Handle 't'/'f' as well as 1/0/True/False
            green_mask_3m = dpr_3m['is_green_payment'].isin(['t', 'T', True, 1])
            nongreen_mask_3m = dpr_3m['is_green_payment'].isin(['f', 'F', False, 0])

            dpr_3m_green = dpr_3m[green_mask_3m]
            dpr_3m_nongreen = dpr_3m[nongreen_mask_3m]

            by_3m_green = dpr_3m_green.groupby('donor_id')['payment_amount']
            by_3m_nongreen = dpr_3m_nongreen.groupby('donor_id')['payment_amount']

            out['gift_count_green_3m'] = by_3m_green.size()
            out['gift_amount_green_3m'] = by_3m_green.sum()
            out['gift_count_nongreen_3m'] = by_3m_nongreen.size()
            out['gift_amount_nongreen_3m'] = by_3m_nongreen.sum()
    
    # --- 12-month window (mid-term) ---
    if not dpr_12m.empty:
        by_12m = dpr_12m.groupby('donor_id')['payment_amount']
        out['gift_count_12m'] = by_12m.size()
        out['gift_amount_12m'] = by_12m.sum()
        out['median_gift_amount_12m'] = by_12m.median()

        if 'is_green_payment' in dpr_12m.columns:
            green_mask_12m = dpr_12m['is_green_payment'].isin(['t', 'T', True, 1])
            nongreen_mask_12m = dpr_12m['is_green_payment'].isin(['f', 'F', False, 0])

            dpr_12m_green = dpr_12m[green_mask_12m]
            dpr_12m_nongreen = dpr_12m[nongreen_mask_12m]

            by_12m_green = dpr_12m_green.groupby('donor_id')['payment_amount']
            by_12m_nongreen = dpr_12m_nongreen.groupby('donor_id')['payment_amount']

            out['gift_count_green_12m'] = by_12m_green.size()
            out['gift_amount_green_12m'] = by_12m_green.sum()
            out['gift_count_nongreen_12m'] = by_12m_nongreen.size()
            out['gift_amount_nongreen_12m'] = by_12m_nongreen.sum()
    
    # --- 36-month window (long-term) ---
    if not dpr_36m.empty:
        by_36m = dpr_36m.groupby('donor_id')['payment_amount']
        out['gift_count_36m'] = by_36m.size()
        out['gift_amount_36m'] = by_36m.sum()
        out['median_gift_amount_36m'] = by_36m.median()

        if 'is_green_payment' in dpr_36m.columns:
            green_mask_36m = dpr_36m['is_green_payment'].isin(['t', 'T', True, 1])
            nongreen_mask_36m = dpr_36m['is_green_payment'].isin(['f', 'F', False, 0])

            dpr_36m_green = dpr_36m[green_mask_36m]
            dpr_36m_nongreen = dpr_36m[nongreen_mask_36m]

            by_36m_green = dpr_36m_green.groupby('donor_id')['payment_amount']
            by_36m_nongreen = dpr_36m_nongreen.groupby('donor_id')['payment_amount']

            out['gift_count_green_36m'] = by_36m_green.size()
            out['gift_amount_green_36m'] = by_36m_green.sum()
            out['gift_count_nongreen_36m'] = by_36m_nongreen.size()
            out['gift_amount_nongreen_36m'] = by_36m_nongreen.sum()
    
    # --- Recency: days since last activity ---
    last_gift_date = dpr_pre_T.groupby('donor_id')['payment_date'].max()
    out['days_since_last_gift'] = (T - last_gift_date).dt.days
    
    # Second-to-last gift: helps detect if donor is becoming less frequent
    # Sort by payment_date descending and use groupby.nth(1)
    second_last = (
        dpr_pre_T
        .sort_values(['donor_id', 'payment_date'], ascending=[True, False])
        .groupby('donor_id')['payment_date']
        .nth(1)  # second row per donor, or NaT if fewer than 2
    )
    
    out['days_since_second_to_last_gift'] = (T - second_last).dt.days
    
    # --- Velocity metrics: are donations accelerating or decelerating? ---
    
    # Recent 3m vs prior 9m (months 3-12)
    amt_0_3 = out.get('gift_amount_3m', pd.Series(0, index=out.index))
    amt_3_12 = (
        dpr_3to12m.groupby('donor_id')['payment_amount'].sum()
        if not dpr_3to12m.empty
        else pd.Series(0, index=out.index)
    )
    # Ratio > 1 means recent giving exceeds prior period
    out['amount_velocity_0to3_vs_3to12'] = amt_0_3 / (amt_3_12 + 1e-6)
    
    # Recent 12m vs prior 24m (months 12-36)
    amt_0_12 = out.get('gift_amount_12m', pd.Series(0, index=out.index))
    amt_12_36 = (
        dpr_12to36m.groupby('donor_id')['payment_amount'].sum()
        if not dpr_12to36m.empty
        else pd.Series(0, index=out.index)
    )
    out['amount_velocity_0to12_vs_12to36'] = amt_0_12 / (amt_12_36 + 1e-6)
    
    # Gift frequency velocity (count-based)
    cnt_0_12 = out.get('gift_count_12m', pd.Series(0, index=out.index))
    cnt_12_36 = (
        dpr_12to36m.groupby('donor_id')['payment_amount'].size()
        if not dpr_12to36m.empty
        else pd.Series(0, index=out.index)
    )
    out['count_velocity_0to12_vs_12to36'] = cnt_0_12 / (cnt_12_36 + 1e-6)
    
    return out

### 2.4 Channel & Payment Type Mix

How donors give: DAF, green payments, gift cards, etc.

In [6]:
def _channel_mix_features(dpr_pre_T, dpr_12m):
    """
    Build features describing how donors give (payment methods, channels).
    
    For each channel, we compute both lifetime and 12-month versions:
    - pct_amount_*: What fraction of $ came through this channel?
    - pct_count_*: What fraction of gifts came through this channel?
    
    Channels covered:
    - DAF (Donor Advised Fund) payments
    - Green payments (environmental offset donations)
    - Gift card purchases
    - Big event donations (e.g., giving days)
    - Optional donation behavior
    - Anonymous gifts
    - Classroom essentials projects
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        All donations before T (for lifetime metrics)
    dpr_12m : pd.DataFrame
        Donations in last 12 months (for recent behavior)
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    out_index = dpr_pre_T['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # --- DAF donations ---
    # DAF = Donor Advised Fund, typically indicates sophisticated/high-capacity donors
    out['pct_amount_daf_lifetime'] = pct_amount(dpr_pre_T, 'daf_payment')
    out['pct_count_daf_lifetime'] = pct_count(dpr_pre_T, 'daf_payment')
    out['pct_amount_daf_12m'] = pct_amount(dpr_12m, 'daf_payment')
    out['pct_count_daf_12m'] = pct_count(dpr_12m, 'daf_payment')
    
    # --- Green payments ---
    # Optional environmental offset donations
    if 'green_payment_amount' in dpr_pre_T.columns:
        by_life = dpr_pre_T.groupby('donor_id')
        by_12 = dpr_12m.groupby('donor_id')
        
        out['pct_amount_green_lifetime'] = (
            by_life['green_payment_amount'].sum()
            / (by_life['payment_amount'].sum() + 1e-6)
        )
        out['pct_amount_green_12m'] = (
            by_12['green_payment_amount'].sum()
            / (by_12['payment_amount'].sum() + 1e-6)
        )
        # v1.3.1: is_green_payment is 't'/'f'; pct_count compares to 1, so v1.2 returned 0 for everyone
        _green_vals = ['t', 'T', True, 1, 'true', 'True', '1']
        out['pct_count_green_lifetime'] = (
            dpr_pre_T['is_green_payment'].isin(_green_vals).groupby(dpr_pre_T['donor_id']).mean())
        out['pct_count_green_12m'] = (
            dpr_12m['is_green_payment'].isin(_green_vals).groupby(dpr_12m['donor_id']).mean())
    
    # --- Gift card purchases ---
    # Donors buying gift cards to give to others
    out['pct_gifts_gift_card_lifetime'] = pct_count(dpr_pre_T, 'gift_card_purchase')
    out['pct_amount_gift_card_lifetime'] = pct_amount(dpr_pre_T, 'gift_card_purchase')
    out['pct_gifts_gift_card_12m'] = pct_count(dpr_12m, 'gift_card_purchase')
    out['pct_amount_gift_card_12m'] = pct_amount(dpr_12m, 'gift_card_purchase')
    
    # --- Big event donations ---
    # Giving days, campaigns, etc.
    out['pct_amount_big_event_lifetime'] = pct_amount(dpr_pre_T, 'payment_on_big_event')
    out['pct_count_big_event_lifetime'] = pct_count(dpr_pre_T, 'payment_on_big_event')
    out['pct_amount_big_event_12m'] = pct_amount(dpr_12m, 'payment_on_big_event')
    out['pct_count_big_event_12m'] = pct_count(dpr_12m, 'payment_on_big_event')
    
    # --- Optional donation rate ---
    # When donors can add optional amounts (e.g., to cover fees)
    out['avg_optional_donation_rate_lifetime'] = (
        dpr_pre_T.groupby('donor_id')['optional_donation_rate'].mean()
    )
    out['avg_optional_donation_rate_12m'] = (
        dpr_12m.groupby('donor_id')['optional_donation_rate'].mean()
    )
    
    # --- Anonymous donations ---
    # Privacy-conscious or humble donors
    out['pct_gifts_anonymous_lifetime'] = pct_count(dpr_pre_T, 'donation_is_anonymous')
    out['pct_gifts_anonymous_12m'] = pct_count(dpr_12m, 'donation_is_anonymous')
    
    # --- Classroom essentials ---
    # Donations to specific project type (basic supplies)
    out['pct_amount_classroom_essentials_lifetime'] = pct_amount(
        dpr_pre_T, 'is_classroom_essentials_list'
    )
    out['pct_amount_classroom_essentials_12m'] = pct_amount(
        dpr_12m, 'is_classroom_essentials_list'
    )
    
    return out

### 2.5 Monthly Giving Program

Subscription/recurring donation features.

In [7]:
def _monthly_features(md_pre_T, dpr_pre_T, dpr_12m, T):
    """
    Monthly program features as of T.

    md_pre_T is payment-level (one row per monthly charge) filtered to joined_date < T.
    Subscription attributes (joined/retired) come from all rows; every amount, recency
    and streak metric uses only charges before T.
    """
    if md_pre_T.empty:
        return pd.DataFrame(index=pd.Index(dpr_pre_T["donor_id"].unique(), name="donor_id"))

    out = pd.DataFrame(index=pd.Index(md_pre_T["donor_id"].unique(), name="donor_id"))

    # --- status at T (subscription-level) ---
    active = (
        (md_pre_T["monthly_subscription_joined_date"] <= T)
        & (md_pre_T["monthly_subscription_retired_date"].isna()
           | (md_pre_T["monthly_subscription_retired_date"] > T))
    )
    out["is_monthly_donor_current"] = active.groupby(md_pre_T["donor_id"]).max().astype(float)

    # --- payments strictly before T ---
    has_charge = "charge_date" in md_pre_T.columns
    pay = md_pre_T[md_pre_T["charge_date"] < T] if has_charge else md_pre_T
    amt_col = "monthly_subscription_payment_amount"

    by_pay = pay.groupby("donor_id")[amt_col]
    out["monthly_lifetime_amount"] = by_pay.sum()
    out["monthly_median_gift_amount"] = by_pay.median()

    if has_charge:
        pay_12m = pay[pay["charge_date"] >= T - pd.DateOffset(months=12)]
        out["monthly_amount_12m"] = pay_12m.groupby("donor_id")[amt_col].sum()
        out["monthly_amount_12m"] = out["monthly_amount_12m"].fillna(
            0.0).where(out["monthly_lifetime_amount"].notna())

        last_charge = pay.groupby("donor_id")["charge_date"].max()
        out["months_since_last_monthly_charge"] = (
            (T.year - last_charge.dt.year) * 12 + (T.month - last_charge.dt.month)
        )

        # Longest run of consecutive calendar months with a charge, before T
        # (replaces monthly_subscription_longest_streak, which is a current-state value)
        ym = pd.DataFrame({
            "donor_id": pay["donor_id"].to_numpy(),
            "ym": (pay["charge_date"].dt.year * 12 + pay["charge_date"].dt.month).to_numpy(),
        }).drop_duplicates().sort_values(["donor_id", "ym"])
        brk = (ym["ym"].diff() != 1) | (ym["donor_id"] != ym["donor_id"].shift())
        ym["run"] = brk.cumsum()
        out["monthly_longest_streak_months"] = (
            ym.groupby(["donor_id", "run"]).size().groupby("donor_id").max()
        )
    else:
        out["monthly_longest_streak_months"] = (
            md_pre_T.groupby("donor_id")["monthly_subscription_longest_streak"].max()
        )

    # --- mix vs project giving (denominator = project dollars, as before) ---
    total_life = dpr_pre_T.groupby("donor_id")["payment_amount"].sum()
    total_12 = dpr_12m.groupby("donor_id")["payment_amount"].sum()
    out["pct_amount_monthly_lifetime"] = out["monthly_lifetime_amount"] / total_life.replace(0, np.nan)
    if "monthly_amount_12m" in out.columns:
        out["pct_amount_monthly_12m"] = out["monthly_amount_12m"] / total_12.replace(0, np.nan)

    # --- months on program (subscription-level, clipped to T) ---
    join_dates = md_pre_T.groupby("donor_id")["monthly_subscription_joined_date"].min()
    retire_dates = md_pre_T.groupby("donor_id")["monthly_subscription_retired_date"].min()
    end_dates = retire_dates.fillna(T).clip(upper=T)
    out["months_on_program"] = (
        (end_dates.dt.year - join_dates.dt.year) * 12 + (end_dates.dt.month - join_dates.dt.month)
    )

    # --- joined monthly before first project gift ---
    fmj = join_dates
    fdd = dpr_pre_T.groupby("donor_id")["payment_date"].min()
    joined_before = (fmj < fdd.reindex(fmj.index)).fillna(False)   # monthly-only donors -> 0, as before
    out["monthly_joined_before_first_project_gift"] = joined_before.reindex(out.index).fillna(False).astype(int)

    return out


### 2.6 Teacher, School & Content Preferences

Loyalty vs diversification patterns.

In [8]:
def _top_entity_share(dpr, key):
    x = dpr[dpr[key].notna() & (dpr[key].astype(str) != "")]
    if x.empty:
        return pd.Series(dtype=float)
    amounts = x.groupby(["donor_id", key])["payment_amount"].sum()
    return amounts.groupby("donor_id").max() / amounts.groupby("donor_id").sum().replace(0, np.nan)


def _teacher_school_features(dpr_pre_T):
    """
    Build features describing donor loyalty vs diversification patterns.
    
    CORRECTED VERSION with:
    - Entropy features set to null/NaN if only one donation
    - New features for pct amount to first/last project gifts
    
    Features measure:
    - Concentration: How focused is giving on specific teachers/schools/categories?
    - Diversification: How many different entities has donor supported?
    - Entropy: Information-theoretic measure of spread
    - Teacher quality: Average metrics of teachers supported
    - Project position: First/last gift preferences
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        Donor Project Records filtered to payment_date < T
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if dpr_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    dpr = dpr_pre_T.copy()
    out_index = dpr['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # --- Count unique entities ---
    out['num_unique_teachers'] = dpr.groupby('donor_id')['teacher_id'].nunique()
    out['num_unique_schools'] = dpr.groupby('donor_id')['school_id'].nunique()
    out['num_unique_categories'] = dpr.groupby('donor_id')['project_category'].nunique()
    out['num_unique_grades'] = dpr.groupby('donor_id')['project_grade'].nunique()
    
    # School ZIP diversity (if available)
    if 'school_zip' in dpr.columns:
        out['num_unique_school_zips'] = dpr.groupby('donor_id')['school_zip'].nunique()
    
    # --- Entropy (diversity) measures ---
    # CORRECTED: Set to NaN if donor only has one donation
    
    # Get gift count per donor
    gift_counts = dpr.groupby('donor_id').size()
    single_gift_donors = gift_counts[gift_counts == 1].index
    
    # Calculate entropy for each dimension
    entropy_teacher = entropy_vectorized(dpr, 'donor_id', 'teacher_id')
    entropy_school = entropy_vectorized(dpr, 'donor_id', 'school_id')
    entropy_category = entropy_vectorized(dpr, 'donor_id', 'project_category')
    entropy_grade = entropy_vectorized(dpr, 'donor_id', 'project_grade')
    
    # Set to NaN for single-gift donors
    out['entropy_teacher'] = entropy_teacher
    out.loc[single_gift_donors, 'entropy_teacher'] = np.nan
    
    out['entropy_school'] = entropy_school
    out.loc[single_gift_donors, 'entropy_school'] = np.nan
    
    out['entropy_category'] = entropy_category
    out.loc[single_gift_donors, 'entropy_category'] = np.nan
    
    out['entropy_grade'] = entropy_grade
    out.loc[single_gift_donors, 'entropy_grade'] = np.nan
    
    if 'school_zip' in dpr.columns:
        entropy_zip = entropy_vectorized(dpr, 'donor_id', 'school_zip')
        out['entropy_zip'] = entropy_zip
        out.loc[single_gift_donors, 'entropy_zip'] = np.nan
    
    # --- Concentration metrics: % of $ going to top entity ---
    # v1.3: denominator excludes rows with no teacher/school/category/grade
    out['pct_amount_to_top_teacher'] = _top_entity_share(dpr, 'teacher_id')
    out['pct_amount_to_top_school'] = _top_entity_share(dpr, 'school_id')
    out['pct_amount_to_top_category'] = _top_entity_share(dpr, 'project_category')
    out['pct_amount_to_top_grade'] = _top_entity_share(dpr, 'project_grade')
    
    # --- Teacher quality metrics ---
    # Average lifetime projects fully funded by teachers this donor supports
    if 'teacher_lifetime_projects_fully_funded' in dpr.columns:
        out['mean_teacher_lifetime_projects_fully_funded'] = (
            dpr.groupby('donor_id')['teacher_lifetime_projects_fully_funded'].mean()
        )
    
    # Average lifetime donations received by teachers this donor supports
    if 'teacher_lifetime_donations' in dpr.columns:
        out['mean_teacher_lifetime_donations'] = (
            dpr.groupby('donor_id')['teacher_lifetime_donations'].mean()
        )
    
    # --- NEW: Project position preferences ---
    # Percent of lifetime amount given to first gifts (gift_is_projects_first = 1)
    if 'gift_is_projects_first' in dpr.columns:
        out['pct_gifts_first_project'] = pct_count(dpr, 'gift_is_projects_first')
        out['pct_amount_first_project'] = pct_amount(dpr, 'gift_is_projects_first')
    
    # Percent of lifetime amount given to last gifts (gift_is_projects_last = 1)
    if 'gift_is_projects_last' in dpr.columns:
        out['pct_gifts_last_project'] = pct_count(dpr, 'gift_is_projects_last')
        out['pct_amount_last_project'] = pct_amount(dpr, 'gift_is_projects_last')
    
    return out

### 2.7 Seasonality Features

In [9]:
def _seasonality_features(dpr_pre_T):
    """
    Build features capturing temporal patterns in giving behavior.
    
    CORRECTED VERSION with:
    - entropy_gift_month set to null/NaN if only one donation
    
    Features created:
    - first_donation_month/quarter: When did they start?
    - first_donation_dow_sin/cos: Day-of-week cyclic encoding
    - pct_amount_in_back_to_school: Aug-Sep giving (7-10% of annual budget)
    - pct_amount_in_final_week_of_year: Dec 24-31 giving (tax planning)
    - pct_amount_on_weekends: Saturday/Sunday behavior
    - pct_amount_in_top_month/quarter: Concentration in favorite period
    - entropy_gift_month: How spread out is giving across months?
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        Donor Project Records filtered to payment_date < T
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if dpr_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    dpr = dpr_pre_T.copy()
    out_index = dpr['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # Extract temporal components
    dpr['gift_month'] = dpr['payment_date'].dt.month
    dpr['gift_quarter'] = dpr['payment_date'].dt.quarter
    dpr['gift_dow'] = dpr['payment_date'].dt.dayofweek  # Monday=0, Sunday=6
    dpr['gift_day'] = dpr['payment_date'].dt.day
    
    # --- First donation temporal features ---
    first_donation = dpr.sort_values(['donor_id', 'payment_date']).groupby('donor_id').first()
    
    out['first_donation_month'] = first_donation['gift_month']
    out['first_donation_quarter'] = first_donation['gift_quarter']
    
    # Cyclic encoding of day-of-week (preserves weekend vs weekday similarity)
    out['first_donation_dow_sin'] = np.sin(2 * np.pi * first_donation['gift_dow'] / 7)
    out['first_donation_dow_cos'] = np.cos(2 * np.pi * first_donation['gift_dow'] / 7)
    
    # --- Seasonal concentration patterns ---
    
    # Back-to-school season (August-September)
    dpr['is_back_to_school'] = dpr['gift_month'].isin([8, 9])
    out['pct_amount_in_back_to_school'] = pct_amount(dpr, 'is_back_to_school')
    
    # Final week of year (tax planning, year-end giving)
    dpr['is_final_week'] = (dpr['gift_month'] == 12) & (dpr['gift_day'] >= 24)
    out['pct_amount_in_final_week_of_year'] = pct_amount(dpr, 'is_final_week')
    
    # Weekend giving (different behavior than weekday)
    dpr['is_weekend'] = dpr['gift_dow'].isin([5, 6])  # Saturday=5, Sunday=6
    out['pct_amount_on_weekends'] = pct_amount(dpr, 'is_weekend')
    
    # --- Peak period concentration ---
    # What % of giving happens in donor's most active month?
    month_amounts = dpr.groupby(['donor_id', 'gift_month'])['payment_amount'].sum()
    top_month_amt = month_amounts.groupby('donor_id').max()
    total_amt = dpr.groupby('donor_id')['payment_amount'].sum()
    out['pct_amount_in_top_month'] = top_month_amt / (total_amt + 1e-9)
    
    # What % of giving happens in donor's most active quarter?
    quarter_amounts = dpr.groupby(['donor_id', 'gift_quarter'])['payment_amount'].sum()
    top_quarter_amt = quarter_amounts.groupby('donor_id').max()
    out['pct_amount_in_top_quarter'] = top_quarter_amt / (total_amt + 1e-9)
    
    # --- Entropy of giving across months ---
    # CORRECTED: Set to NaN if donor only has one donation
    gift_counts = dpr.groupby('donor_id').size()
    single_gift_donors = gift_counts[gift_counts == 1].index
    
    entropy_month = entropy_vectorized(dpr, 'donor_id', 'gift_month')
    out['entropy_gift_month'] = entropy_month
    out.loc[single_gift_donors, 'entropy_gift_month'] = np.nan
    
    return out

### 2.8 Email Engagement

In [10]:
def _email_features(email_pre_T, email_3m, email_12m, T):
    """
    Build email engagement features from 12-month email summary data.
    
    Features created:
    - emails_sent/opened/clicked for 3m and 12m windows
    - open_rate, click_rate (clicks per email sent)
    - email_open_rate_velocity: recent vs longer-term trend
    - days_since_last_email_sent: recency
    
    Note: This uses monthly aggregated data, so recency is approximate
    
    Parameters
    ----------
    email_pre_T : pd.DataFrame
        All email events before T
    email_3m, email_12m : pd.DataFrame
        Email events in respective windows
    T : pd.Timestamp
        Reference time
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if email_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    def email_agg(df):
        """Aggregate email metrics for a given window"""
        if df.empty:
            idx = email_pre_T['donor_id'].unique()
            zero = pd.Series(0, index=idx)
            return zero, zero, zero, zero, zero
        
        by = df.groupby('donor_id')
        sent = by['email_sent_count'].sum()
        opened = by['email_open_count'].sum()
        clicked = by['email_click_count'].sum()
        
        # Rates: opens/clicks per email sent
        open_rate = opened / (sent + 1e-6)
        click_rate = clicked / (sent + 1e-6)
        
        return sent, opened, clicked, open_rate, click_rate
    
    # Aggregate for both windows
    sent_3, open_3, click_3, or_3, cr_3 = email_agg(email_3m)
    sent_12, open_12, click_12, or_12, cr_12 = email_agg(email_12m)
    
    idx = email_pre_T['donor_id'].unique()
    out = pd.DataFrame(index=idx)
    
    # 3-month window
    out['emails_sent_3m'] = sent_3
    out['emails_opened_3m'] = open_3
    out['emails_clicked_3m'] = click_3
    out['email_open_rate_3m'] = or_3
    out['email_click_rate_3m'] = cr_3
    
    # 12-month window
    out['emails_sent_12m'] = sent_12
    out['emails_opened_12m'] = open_12
    out['emails_clicked_12m'] = click_12
    out['email_open_rate_12m'] = or_12
    out['email_click_rate_12m'] = cr_12
    
    # Velocity: is engagement improving or declining?
    # Positive = recent engagement higher than long-term average
    out['email_open_rate_velocity_3m_vs_12m'] = or_3 - or_12
    
    # Recency (approximate, since data is monthly)
    last_email_month = email_pre_T.groupby('donor_id')['email_month_start'].max()
    out['days_since_last_email_sent'] = (T - last_email_month).dt.days
    
    # Note: email type mix features would go here if you have
    # a mapping from email_type to type_group (appeal, newsletter, etc.)
    
    return out

### 2.9 Site Behavior

In [11]:
SITE_ATTR_COLS = {
    "share_email": "site_attr_email_3m",
    "share_ad": "site_attr_ad_3m",
    "share_sharetools": "site_attr_sharetools_3m",
    "share_unattributed": "site_attr_unattributed_3m",
    "share_other": "site_attr_other_3m",
}


def _site_features(site_pre_T, site_3m, T):
    """
    On-site engagement features from daily donor aggregates.

    Row grain: one row per donor per activity_date. sessions_day, duration_sum_min,
    time_to_checkout_sum_min and *_visits_day are daily totals; share_* columns are
    treated as within-day session shares and weighted by sessions_day (hypothesis:
    confirm with the site extract definition).
    """
    if site_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name="donor_id"))

    out = pd.DataFrame(index=pd.Index(site_pre_T["donor_id"].unique(), name="donor_id"))

    if not site_3m.empty:
        x = site_3m
        if "sessions_day" in x.columns:
            sess = pd.to_numeric(x["sessions_day"], errors="coerce").fillna(0)
        else:
            sess = pd.Series(1.0, index=x.index)   # one session per active day
        g = pd.DataFrame({"donor_id": x["donor_id"].to_numpy(), "sessions": sess.to_numpy()})
        for src_col in ["duration_sum_min", "time_to_checkout_sum_min", "cart_visits_day",
                        "project_page_visits_day", "teacher_page_visits_day", "search_visits_day",
                        "came_from_campaign"]:
            if src_col in x.columns:
                g[src_col] = pd.to_numeric(x[src_col], errors="coerce").fillna(0).to_numpy()
        for share_col in SITE_ATTR_COLS:
            if share_col in x.columns:
                g[share_col] = (pd.to_numeric(x[share_col], errors="coerce").fillna(0) * sess).to_numpy()
        g["active_day"] = 1

        agg = g.groupby("donor_id").sum()
        sessions = agg["sessions"].replace(0, np.nan)

        out["days_with_any_site_activity_3m"] = agg["active_day"]
        out["avg_sessions_per_active_day_3m"] = agg["sessions"] / agg["active_day"]
        if "duration_sum_min" in agg:
            out["avg_session_duration_min_3m"] = agg["duration_sum_min"] / sessions
        if "cart_visits_day" in agg:
            out["checkout_intent_min_per_session_3m"] = agg["cart_visits_day"] / sessions
        if "time_to_checkout_sum_min" in agg:
            out["time_to_checkout_min_per_session_3m"] = agg["time_to_checkout_sum_min"] / sessions
        if "came_from_campaign" in agg:
            out["campaign_session_share_3m"] = agg["came_from_campaign"] / agg["active_day"]

        page_cols = ["project_page_visits_day", "teacher_page_visits_day", "search_visits_day"]
        if all(c in agg for c in page_cols):
            tot = agg[page_cols].sum(axis=1).replace(0, np.nan)
            out["share_project_page_session_pct_3m"] = agg["project_page_visits_day"] / tot
            out["share_teacher_page_session_pct_3m"] = agg["teacher_page_visits_day"] / tot
            out["share_search_page_session_pct_3m"] = agg["search_visits_day"] / tot

        for share_col, name in SITE_ATTR_COLS.items():
            if share_col in agg:
                out[name] = agg[share_col] / sessions

        if "device_type" in x.columns:
            dev = x["device_type"].astype(str).str.lower()
            dc = pd.crosstab(x["donor_id"], dev)
            tot_dev = dc.sum(axis=1)
            for dv in ["mobile", "desktop", "tablet"]:
                out[f"device_share_{dv}_3m"] = (dc[dv] / tot_dev) if dv in dc.columns else 0.0

    if "cart_visits_day" in site_pre_T.columns:
        has_cart = site_pre_T[pd.to_numeric(site_pre_T["cart_visits_day"], errors="coerce") > 0]
        if not has_cart.empty:
            out["days_since_last_cart_visit"] = (T - has_cart.groupby("donor_id")["activity_date"].max()).dt.days

    return out


### 2.10 Share Events

In [12]:
def _share_features(share_pre_T, share_12m, T):
    """
    Build social sharing behavior features.
    
    Features created:
    - share_events_lifetime, share_events_12m: Volume
    - share_active_months_12m: Frequency
    - share_gap_mean/cv_days: Sharing rhythm
    - share_month_coverage_ratio: Consistency over tenure
    
    Parameters
    ----------
    share_pre_T : pd.DataFrame
        All share events before T
    share_12m : pd.DataFrame
        Share events in last 12 months
    T : pd.Timestamp
        Reference time
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if share_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    out_index = share_pre_T['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # --- Volume metrics ---
    out['share_events_lifetime'] = (
        share_pre_T.groupby('donor_id')['share_event_count'].sum()
    )
    out['share_events_12m'] = (
        share_12m.groupby('donor_id')['share_event_count'].sum()
        if not share_12m.empty else 0
    )
    
    # --- Active months (consistency) ---
    if not share_12m.empty:
        active_months_12m = (
            share_12m[share_12m['share_event_count'] > 0]
            .groupby('donor_id')['share_month_start']
            .nunique()
        )
        out['share_active_months_12m'] = active_months_12m
    
    # --- Sharing rhythm: gaps between share months ---
    def gap_stats(s):
        if s.shape[0] < 2:
            return pd.Series({
                'share_gap_mean_days': np.nan,
                'share_gap_cv_days': np.nan
            })
        # Sort by time, then diff
        gaps = s.sort_values().diff().dropna().dt.days
        mean = gaps.mean()
        cv = gaps.std() / (mean + 1e-6)
        return pd.Series({
            'share_gap_mean_days': mean,
            'share_gap_cv_days': cv
        })

    gap_df = (
        share_pre_T.groupby('donor_id')['share_month_start']
        .apply(gap_stats)
        .unstack()          # columns: share_gap_mean_days, share_gap_cv_days
    )

    out = out.join(gap_df, how='left')

    # --- Coverage ratio ---
    first_share = share_pre_T.groupby('donor_id')['share_month_start'].min()
    last_share = share_pre_T.groupby('donor_id')['share_month_start'].max()
    
    # v1.3.1: share of months between first and last share (inclusive) with >= 1 share
    # event. v1.2 divided 12-month active months by the lifetime span, mixing windows.
    span_months = (
        (last_share.dt.year - first_share.dt.year) * 12
        + (last_share.dt.month - first_share.dt.month) + 1
    )
    active_months_life = (
        share_pre_T[share_pre_T['share_event_count'] > 0]
        .groupby('donor_id')['share_month_start'].nunique()
    )
    out['share_month_coverage_ratio'] = active_months_life.reindex(span_months.index) / span_months

    return out

### 2.11 Same Schoool & Teacher Available Flags

In [13]:
def _future_opportunity_features(dpr_pre_T, df_project_dates, T, H):
    """
    Opportunity features for the donor's TOP (most-funded) school and teacher.

    TEMPORAL CONTRACT: uses only project rows posted strictly before T, and only
    date fields whose values are known as of T. H is accepted for signature
    compatibility and is deliberately unused -- nothing here reads [T, T+H).

    Columns
    -------
    top_{school,teacher}_open_project_at_T    1 if accepting donations at T
    top_{school,teacher}_projects_posted_12m  count posted in [T-365d, T)
    top_{school,teacher}_days_since_posted    days from last pre-T posting to T
                                              (sentinel 9999 if never)
    """
    NEVER_POSTED = 9999

    out_cols = [
        'top_school_open_project_at_T', 'top_teacher_open_project_at_T',
        'top_school_projects_posted_12m', 'top_teacher_projects_posted_12m',
        'top_school_days_since_posted', 'top_teacher_days_since_posted',
    ]

    if dpr_pre_T is None or dpr_pre_T.empty:
        return pd.DataFrame(columns=out_cols, index=pd.Index([], name='donor_id'))

    donor_index = pd.Index(dpr_pre_T['donor_id'].unique(), name='donor_id')
    out = pd.DataFrame(index=donor_index)

    def _empty_defaults():
        for c in out_cols:
            out[c] = NEVER_POSTED if c.endswith('days_since_posted') else 0
        return out[out_cols]

    if df_project_dates is None or df_project_dates.empty:
        return _empty_defaults()

    T = pd.to_datetime(T)
    proj = df_project_dates.copy()
    for c in ['project_last_posted_date', 'project_funded_date', 'project_expiration_date']:
        if c in proj.columns:
            proj[c] = pd.to_datetime(proj[c], errors='coerce')
        else:
            proj[c] = pd.NaT

    # -- GATE 1: only projects that exist and are visible as of T -------------
    proj = proj[proj['project_last_posted_date'] < T]
    if proj.empty:
        return _empty_defaults()

    # -- GATE 2: only outcomes realized before T ------------------------------
    # Funding is a future event. At T we know a project was funded only if the
    # funded date has already passed; anything funded later must look open.
    funded_by_T = proj['project_funded_date'].notna() & (proj['project_funded_date'] < T)
    not_expired_at_T = proj['project_expiration_date'].isna() | (proj['project_expiration_date'] >= T)
    proj['open_at_T'] = (~funded_by_T) & not_expired_at_T

    proj['posted_12m'] = proj['project_last_posted_date'] >= (T - pd.Timedelta(days=365))

    def _top_entity(col):
        if not {col, 'payment_amount'}.issubset(dpr_pre_T.columns):
            return None
        agg = dpr_pre_T.groupby(['donor_id', col], observed=True)['payment_amount'].sum().reset_index()
        return (agg.sort_values(['donor_id', 'payment_amount'], ascending=[True, False])
                   .groupby('donor_id', observed=True).first().reset_index()[['donor_id', col]])

    for entity, prefix in (('school_id', 'top_school'), ('teacher_id', 'top_teacher')):
        top = _top_entity(entity) if entity in proj.columns else None
        if top is None or top.empty:
            out[f'{prefix}_open_project_at_T'] = 0
            out[f'{prefix}_projects_posted_12m'] = 0
            out[f'{prefix}_days_since_posted'] = NEVER_POSTED
            continue

        grp = proj.groupby(entity, observed=True)
        ent_open = grp['open_at_T'].any().astype(int)
        ent_n12m = grp['posted_12m'].sum().astype(float)
        ent_last = (T - grp['project_last_posted_date'].max()).dt.days.astype(float)

        key = top.set_index('donor_id')[entity]
        out[f'{prefix}_open_project_at_T'] = key.map(ent_open).reindex(donor_index).fillna(0).astype(int)
        out[f'{prefix}_projects_posted_12m'] = key.map(ent_n12m).reindex(donor_index).fillna(0.0)
        out[f'{prefix}_days_since_posted'] = key.map(ent_last).reindex(donor_index).fillna(NEVER_POSTED)

    return out[out_cols]

### 2.11 Project Outcomes & Matching

In [14]:
def _project_outcome_features(dpr_pre_T):
    """
    Build features describing project success and matching behavior.
    
    CORRECTED VERSION with:
    - mean_match_multiplier: subtract 1 from all values, impute 0 for records with optional_donation_rate
    
    Features created:
    - pct_projects_fully_funded: Success rate of projects supported
    - pct_gifts_with_match: How often donations are matched
    - mean_match_multiplier: Average EXCESS match (1.5 becomes 0.5), with 0s for optional donations
    - mean/median_project_total_cost: Scale of projects supported
    - median_donor_to_project_distance_mi: Geographic preference
    - pct_gifts_within_15mi: Local giving behavior
    - is_local_donor: Predominantly supports nearby schools
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        Donor Project Records filtered to payment_date < T
    T : pd.Timestamp
        Reference time
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id
    """
    if dpr_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    dpr = dpr_pre_T.copy()
    out_index = dpr['donor_id'].unique()
    out = pd.DataFrame(index=out_index)
    
    # --- Project success metrics ---
    if 'project_got_fully_funded' in dpr.columns:
        out['pct_projects_fully_funded'] = (
            dpr.groupby('donor_id')['project_got_fully_funded'].mean()
        )
    elif 'project_fully_funded' in dpr.columns:
        out['pct_projects_fully_funded'] = (
            dpr.groupby('donor_id')['project_fully_funded'].mean()
        )
    
    # --- Matching behavior ---
    # 1. Subtract 1 from all match_xyi_multiplier values (so 1.5 becomes 0.5)
    # 2. Impute 0 for records that have a value in optional_donation_rate
    
    if 'match_xyi_multiplier' in dpr.columns:
        # Step 1: Filter to rows with optional_donation_rate present
        dpr_with_optional = dpr[dpr['optional_donation_rate'].notna()].copy()
        
        if not dpr_with_optional.empty:
            # Step 2: Calculate match_excess (subtract 1)
            dpr_with_optional['match_excess'] = dpr_with_optional['match_xyi_multiplier'] - 1.0
            
            # Step 3: Fill nulls in match_excess with 0
            dpr_with_optional['match_excess'] = dpr_with_optional['match_excess'].fillna(0)
            
            # Step 4: Calculate mean per donor
            out['mean_match_multiplier'] = (
                dpr_with_optional.groupby('donor_id')['match_excess'].mean()
            )
        else:
            out['mean_match_multiplier'] = np.nan
        
        # Percent of gifts that received any match (before adjustments)
        had_match = dpr['match_xyi_multiplier'] > 1.0
        out['pct_gifts_with_match'] = (
            dpr.groupby('donor_id')
            .apply(lambda x: had_match.loc[x.index].mean())
        )
    
    # --- Project cost metrics ---
    out['mean_project_total_cost'] = dpr.groupby('donor_id')['project_total_cost'].mean()
    out['median_project_total_cost'] = dpr.groupby('donor_id')['project_total_cost'].median()
    
    # --- Geographic patterns ---
    if 'distance_mi' in dpr.columns:
        out['median_donor_to_project_distance_mi'] = (
            dpr.groupby('donor_id')['distance_mi'].median()
        )
        
        # Local giving: within 15 miles
        dpr['is_within_15mi'] = dpr['distance_mi'] <= 15
        out['pct_gifts_within_15mi'] = (
            dpr.groupby('donor_id')['is_within_15mi'].mean()
        )
        
        # Predominantly local donor (>75% of gifts within 15 miles)
        out['is_local_donor'] = (out['pct_gifts_within_15mi'] > 0.75).astype(int)
    
    return out

### 2.12 Latest Donation

In [15]:
def _latest_donation_features(dpr_pre_T, df_share, T):
    """
    Extract features from each donor's most recent donation before time T.
    
    This captures the "state" of the donor at their last interaction,
    which can be highly predictive of near-term behavior.
    
    Features include:
    - Amount split by green vs non-green
    - Project characteristics (cost, category, grade, fully funded status)
    - Teacher metrics
    - Payment type and optional donation behavior
    - Geographic distance
    - Referral channel (how they arrived)
    - Social sharing behavior in the month of latest donation
    
    Parameters
    ----------
    dpr_pre_T : pd.DataFrame
        Donor Project Records filtered to payment_date < T
        Expected columns: donor_id, payment_date, payment_amount, 
        is_green_payment, project_total_cost, project_got_fully_funded,
        teacher_lifetime_projects_fully_funded, gift_is_projects_first,
        gift_is_projects_last, optional_donation_rate, payment_type,
        project_category, project_grade, referral_source, 
        referral_medium, donor_lat_long, school_lat_long
    df_share : pd.DataFrame
        Share events with columns: donor_id, share_sent_month
        share_sent_month format: "YYYY-MM" (e.g., "2022-07")
    T : pd.Timestamp
        Reference time (as-of date)
        
    Returns
    -------
    features : pd.DataFrame
        Feature matrix indexed by donor_id with 'latest_*' columns
    """
    
    if dpr_pre_T.empty:
        return pd.DataFrame(index=pd.Index([], name='donor_id'))
    
    # =========================================================================
    # Get most recent donation per donor
    # =========================================================================
    
    # v1.3.3: take the last actual row. groupby().last() took the last non-null value of
    # each column separately, which could combine fields from different rows. Same-day ties
    # break on donation_n / payment_id (stable sort; nulls first so a real value wins).
    sort_cols = ['donor_id', 'payment_date'] + [
        c for c in ('donation_n', 'payment_id') if c in dpr_pre_T.columns
    ]
    latest = (
        dpr_pre_T
        .sort_values(sort_cols, kind='mergesort', na_position='first')
        .groupby('donor_id')
        .tail(1)
        .set_index('donor_id')
    )
    
    out = pd.DataFrame(index=latest.index)
    
    # =========================================================================
    # Amount features: split by green vs non-green
    # =========================================================================
    
    # is_green_payment is string "t" or "f"
    is_green = latest['is_green_payment'].isin(['t', 'T', True, 1])
    
    out['latest_gift_amount_green'] = np.where(
        is_green, 
        latest['payment_amount'], 
        0
    )
    
    out['latest_gift_amount_nongreen'] = np.where(
        ~is_green,
        latest['payment_amount'],
        0
    )
    
    # =========================================================================
    # Project characteristics
    # =========================================================================
    
    out['latest_match_xyi_multiplier'] = latest.get('match_xyi_multiplier', np.nan)
    out['latest_project_total_cost'] = latest['project_total_cost']
    out['latest_project_got_fully_funded'] = latest['project_got_fully_funded'].fillna(0).astype(int)
    
    # Teacher metrics at time of latest donation
    out['latest_teacher_lifetime_projects_fully_funded'] = latest.get(
        'teacher_lifetime_projects_fully_funded', np.nan
    )
    
    # Position in project funding sequence
    out['latest_donation_is_projects_first'] = latest['gift_is_projects_first'].fillna(0).astype(int)
    out['latest_donation_is_projects_last'] = latest['gift_is_projects_last'].fillna(0).astype(int)
    
    # =========================================================================
    # Payment type and behavior
    # =========================================================================
    
    out['latest_payment_type_is_green'] = is_green.astype(int)
    
    # Optional donation as percentage of total
    out['latest_optional_donation_percent'] = latest.get('optional_donation_rate', 0)
    
    # Gift card purchase flag
    if 'gift_card_purchase' in latest.columns:
        out['latest_is_giftcard_purchase'] = (
            latest['gift_card_purchase'].fillna(0).astype(int)
        )
    else:
        out['latest_is_giftcard_purchase'] = 0
    
    # =========================================================================
    # Categorical features (for one-hot encoding downstream)
    # =========================================================================
    
    out['latest_project_grade'] = latest['project_grade']
    out['latest_project_category'] = latest['project_category']
    
    # =========================================================================
    # Referral channel (same logic as Repeat Donor Behaviors)
    # =========================================================================
    
    valid_media = {
        'email', 'directlink', 'facebook', 'nextdoor',
        'sharetray', 'mobilesharetray', 'page', 'ig', 'sendfriend'
    }
    valid_sources = {'email', 'unattributed', 'sharetools', 'ad', 'other'}  # v1.3: match SQL buckets
    
    # If referral_medium is in valid_media, use it; otherwise use referral_source
    # Treat non-dc/google sources as "Oth"
    referral_source_clean = latest.get('referral_source', pd.Series(index=latest.index))
    referral_source_clean = referral_source_clean.fillna('unattributed')
    referral_source_clean = referral_source_clean.where(
        referral_source_clean.isin(valid_sources),
        'other'
    )
    
    referral_medium = latest.get('referral_medium', pd.Series(index=latest.index))
    
    out['latest_referral_channel'] = np.where(
        referral_medium.isin(valid_media),
        referral_medium,
        referral_source_clean
    )
    
    # =========================================================================
    # Distance to school
    # =========================================================================
    
    if 'distance_mi' in latest.columns:
        out['latest_distance_mi'] = latest['distance_mi']
    else:
        out['latest_distance_mi'] = np.nan
    
    # =========================================================================
    # Social sharing in month of latest donation
    # =========================================================================
    
    if df_share is not None and not df_share.empty:
        # Extract year-month from latest donation date
        latest_month = pd.to_datetime(latest['payment_date']).dt.to_period('M').astype(str)
        
        # Create a mapping of donor_id -> set of months they shared
        share_months = (
            df_share
            .groupby('donor_id')['share_sent_month']
            .apply(set)
            .to_dict()
        )
        
        # For each donor, check if they shared in their latest donation month
        out['latest_shared_any'] = out.index.map(
            lambda donor_id: int(
                latest_month.loc[donor_id] in share_months.get(donor_id, set())
                if donor_id in latest_month.index
                else 0
            )
        )
    else:
        out['latest_shared_any'] = 0
    
    return out

### 2.13 Behavioral Expansion (v1.3)

Occasions, funding depth, channel and payment mix, events, urgency, school need, tips, STY, email type mix, lifecycle, and the as-of-T cohort rule.

In [16]:
# =============================================================================
# 2.13 BEHAVIORAL EXPANSION (v1.3)
#
# Uses the new DPR columns when present (see dpr_extract_changes.sql) and falls
# back to the current extract otherwise:
#   cart_id                      -> donation_n (donor cart sequence) -> donor-day
#   gift_to_project_cost_ratio   -> payment_amount / project_total_cost
#   days_gift_to_expiration      -> project_expiration_date - payment_date
#   payment_on_giving_tuesday    -> computed from payment_date
#   has_school / NULL equity     -> equity flags masked where school_id is null
#
# Conventions: shares are NaN when the denominator is empty (no epsilon); counts
# are 0 when the donor has history but none in the window. Windows use
# DateOffset(months=...) to match build_features.
# =============================================================================

import numpy as np
import pandas as pd

TRUE_VALS = [1, True, "t", "T", "true", "True", "1"]
REFERRAL_SOURCES = ["email", "unattributed", "sharetools", "ad", "other"]
PAYMENT_TYPES = ["credit", "paypal", "amazon", "check", "giftcard"]
NON_DAF_CHECK_TYPES = ["IRA", "FOUNDATION", "INDIVIDUAL", "ACH"]
EXPIRING_SOON_DAYS = 14            # analyst choice; tune
SUMMER_MONTHS = (6, 7)             # Jun-Jul; Aug-Sep is already back-to-school


def _empty_frame():
    return pd.DataFrame(index=pd.Index([], name="donor_id"))


def _giving_tuesdays(years):
    """Tuesday after US Thanksgiving (4th Thursday of November)."""
    out = []
    for y in years:
        nov1 = pd.Timestamp(year=int(y), month=11, day=1)
        first_thu = nov1 + pd.Timedelta(days=(3 - nov1.weekday()) % 7)
        out.append(first_thu + pd.Timedelta(days=26))
    return pd.DatetimeIndex(out)


def _as_datetime(s):
    """Return s as datetime64 without modifying the caller's frame."""
    return s if pd.api.types.is_datetime64_any_dtype(s) else pd.to_datetime(s, errors="coerce")


def _occasion_key(d):
    """
    Checkout key per row, cascading per row: cart_id -> donation_n -> donor-day.
    Tiers occupy disjoint ranges so keys never collide within a donor:
      cart_id >= 0 | donation_n -> -1e7 - n (<= -1e7) | donor-day -> -(days since 1970) - 1 (> -1e6)
    """
    # days since epoch; independent of datetime resolution (ns in pandas 2, may be us in pandas 3)
    day_ord = (d["payment_date"].dt.normalize() - pd.Timestamp("1970-01-01")).dt.days.astype(float)
    key = -day_ord - 1.0
    if "donation_n" in d.columns:
        dn = pd.to_numeric(d["donation_n"], errors="coerce")
        key = (-1e7 - dn).where(dn.notna(), key)
    if "cart_id" in d.columns:
        cid = pd.to_numeric(d["cart_id"], errors="coerce")
        key = cid.where(cid.notna(), key)
    return key


def _yes_no(s):
    return s.map({"Yes": 1.0, "No": 0.0, "yes": 1.0, "no": 0.0})


def _grouped_mean(values, donor_ids, mask=None):
    df = pd.DataFrame({"donor_id": donor_ids, "v": values})
    if mask is not None:
        df = df[np.asarray(mask)]
    return df.groupby("donor_id")["v"].mean()


# -----------------------------------------------------------------------------
# 1. Project-giving behavior (occasions, funding depth, channel, timing, equity, tips)
# -----------------------------------------------------------------------------
def _expansion_features(dpr_pre_T, T):
    if dpr_pre_T is None or dpr_pre_T.empty:
        return _empty_frame()

    d = dpr_pre_T
    donor = d["donor_id"].to_numpy()
    pdate = d["payment_date"]
    day = pdate.dt.normalize()
    w12 = (pdate >= T - pd.DateOffset(months=12)).to_numpy()
    w36 = (pdate >= T - pd.DateOffset(months=36)).to_numpy()
    cut_2yr = T - pd.DateOffset(months=24)

    out = pd.DataFrame(index=pd.Index(pd.unique(donor), name="donor_id"))

    # ---------- occasions (one checkout = one occasion) ----------
    s = pd.DataFrame({
        "donor_id": donor, "day": day.to_numpy(), "occ": _occasion_key(d).to_numpy(),
        "ym": (pdate.dt.year * 12 + pdate.dt.month).to_numpy(), "w12": w12, "w36": w36,
    })
    occ_first = s.sort_values(["donor_id", "day"]).drop_duplicates(["donor_id", "occ"])

    n_occ = occ_first.groupby("donor_id").size()
    out["n_occasions_lifetime"] = n_occ
    out["n_occasions_12m"] = occ_first[occ_first["w12"]].groupby("donor_id").size()
    out["n_occasions_36m"] = occ_first[occ_first["w36"]].groupby("donor_id").size()
    for c in ["n_occasions_12m", "n_occasions_36m"]:
        out[c] = out[c].fillna(0)

    out["gifts_per_occasion_lifetime"] = s.groupby("donor_id").size() / n_occ
    rows12 = s[s["w12"]].groupby("donor_id").size()
    out["gifts_per_occasion_12m"] = rows12 / out["n_occasions_12m"].replace(0, np.nan)

    # giving-day rhythm (distinct days; replaces the 0-day gaps the row-level gap features see)
    days = s[["donor_id", "day"]].drop_duplicates()
    out["n_giving_days_lifetime"] = days.groupby("donor_id").size()
    d2 = days[days["day"] >= cut_2yr].sort_values(["donor_id", "day"])
    gaps = pd.DataFrame({
        "donor_id": d2["donor_id"].to_numpy(),
        "gap": d2.groupby("donor_id")["day"].diff().dt.days.to_numpy(),
    }).dropna()
    g_mean = gaps.groupby("donor_id")["gap"].mean()
    out["giving_day_gap_mean_days_last2yr"] = g_mean
    out["giving_day_gap_cv_last2yr"] = gaps.groupby("donor_id")["gap"].std() / g_mean

    ds = days.sort_values(["donor_id", "day"], ascending=[True, False])
    ds = ds.assign(rk=ds.groupby("donor_id").cumcount())
    prior_day = ds[ds["rk"] == 1].set_index("donor_id")["day"]
    out["days_since_prior_giving_day"] = (T - prior_day).dt.days

    out["n_active_months_12m"] = s[s["w12"]].groupby("donor_id")["ym"].nunique()
    out["n_active_months_36m"] = s[s["w36"]].groupby("donor_id")["ym"].nunique()
    for c in ["n_active_months_12m", "n_active_months_36m"]:
        out[c] = out[c].fillna(0)
    out["occasions_per_active_month_36m"] = out["n_occasions_36m"] / out["n_active_months_36m"].replace(0, np.nan)

    # ---------- funding depth (project rows only) ----------
    amt = pd.to_numeric(d["payment_amount"], errors="coerce")
    if "gift_to_project_cost_ratio" in d.columns:
        ratio = pd.to_numeric(d["gift_to_project_cost_ratio"], errors="coerce")
    else:
        cost = pd.to_numeric(d.get("project_total_cost"), errors="coerce")
        ratio = amt / cost.where(cost > 0)
    has_proj = ratio.notna()
    r = ratio.to_numpy()
    half = np.where(has_proj, (ratio >= 0.5).astype(float), np.nan)
    full = np.where(has_proj, (ratio >= 0.99).astype(float), np.nan)

    f = pd.DataFrame({"donor_id": donor, "r": r, "half": half, "full": full, "w12": w12})
    gl, g12 = f.groupby("donor_id"), f[f["w12"]].groupby("donor_id")
    out["coverage_project_cost_lifetime"] = _grouped_mean(has_proj.astype(float).to_numpy(), donor)
    out["median_gift_to_cost_ratio_lifetime"] = gl["r"].median()
    out["median_gift_to_cost_ratio_12m"] = g12["r"].median()
    out["pct_gifts_over_half_cost_lifetime"] = gl["half"].mean()
    out["pct_gifts_over_half_cost_12m"] = g12["half"].mean()
    out["pct_gifts_full_cost_lifetime"] = gl["full"].mean()

    if "gift_completed_project_funding" in d.columns:
        cf = pd.to_numeric(d["gift_completed_project_funding"], errors="coerce").astype(float)
        cf = cf.where(has_proj)
        out["pct_gifts_completed_funding_lifetime"] = _grouped_mean(cf.to_numpy(), donor)
        out["pct_gifts_completed_funding_12m"] = _grouped_mean(cf.to_numpy(), donor, w12)
        num = pd.DataFrame({"donor_id": donor, "a": (amt * cf).to_numpy(), "b": amt.where(has_proj).to_numpy()})
        sums = num.groupby("donor_id")[["a", "b"]].sum(min_count=1)
        out["pct_amount_completed_funding_lifetime"] = sums["a"] / sums["b"].replace(0, np.nan)

    # ---------- channel (occasion level: first row of each checkout) ----------
    # Assumes referral source is constant within a checkout (0.00% of checkouts mix
    # sources on the current extract; re-checked by verify_expansion_inputs).
    o_idx = occ_first.index
    if "referral_source" in d.columns:
        src = d["referral_source"].astype("object").fillna("unattributed").str.lower().to_numpy()[o_idx]
        occ_don = occ_first["donor_id"].to_numpy()
        occ_w12 = occ_first["w12"].to_numpy()
        oh = pd.DataFrame({v: (src == v).astype(float) for v in REFERRAL_SOURCES})
        oh["donor_id"] = occ_don
        life = oh.groupby("donor_id")[REFERRAL_SOURCES].mean()
        twelve = oh[occ_w12].groupby("donor_id")[REFERRAL_SOURCES].mean()
        out = out.join(life.add_prefix("pct_occasions_src_").add_suffix("_lifetime"))
        out = out.join(twelve.add_prefix("pct_occasions_src_").add_suffix("_12m"))

        cnt = oh.groupby("donor_id")[REFERRAL_SOURCES].sum()
        p = cnt.div(cnt.sum(axis=1), axis=0)
        ent = -(p * np.log(p.where(p > 0))).sum(axis=1)
        ent = ent / np.log(len(REFERRAL_SOURCES))
        out["entropy_referral_source_norm_lifetime"] = ent.where(cnt.sum(axis=1) >= 2)
        out["entry_referral_source"] = pd.Series(src, index=occ_don).groupby(level=0).first()

    if "payment_type" in d.columns:
        # gift level: 4.71% of checkouts mix payment types (e.g. gift card + card)
        pt = d["payment_type"].astype("object").str.lower().to_numpy()
        ph = pd.DataFrame({v: (pt == v).astype(float) for v in PAYMENT_TYPES})
        ph["donor_id"] = donor
        out = out.join(ph.groupby("donor_id")[PAYMENT_TYPES].mean()
                       .add_prefix("pct_gifts_paytype_").add_suffix("_lifetime"))

    if "received_check_type" in d.columns:
        ct = d["received_check_type"].astype("object").str.upper()
        nondaf = ct.isin(NON_DAF_CHECK_TYPES).astype(float).to_numpy()
        out["has_nondaf_check_vehicle_lifetime"] = (
            pd.Series(nondaf, index=donor).groupby(level=0).max()
        )

    # ---------- seasonality / events (occasion level) ----------
    occ_day = pd.DatetimeIndex(occ_first["day"].to_numpy())
    if "payment_on_giving_tuesday" in d.columns:
        gt = pd.to_numeric(d["payment_on_giving_tuesday"], errors="coerce").to_numpy()[o_idx].astype(float)
    else:
        gt_days = _giving_tuesdays(range(occ_day.year.min(), occ_day.year.max() + 1))
        gt = occ_day.isin(gt_days).astype(float)
    ev = pd.DataFrame({
        "donor_id": occ_first["donor_id"].to_numpy(),
        "gt": gt,
        "q4": (occ_day.quarter == 4).astype(float),
        "summer": occ_day.month.isin(SUMMER_MONTHS).astype(float),
    })
    evm = ev.groupby("donor_id")[["gt", "q4", "summer"]].mean()
    out["pct_occasions_giving_tuesday_lifetime"] = evm["gt"]
    out["pct_occasions_q4_lifetime"] = evm["q4"]
    out["pct_occasions_summer_lifetime"] = evm["summer"]

    # ---------- urgency (project rows with an expiration date) ----------
    if "days_gift_to_expiration" in d.columns:
        dte = pd.to_numeric(d["days_gift_to_expiration"], errors="coerce")
    elif "project_expiration_date" in d.columns:
        exp = pd.to_datetime(d["project_expiration_date"], format="%Y-%m-%d", errors="coerce")
        dte = (exp - day).dt.days
    else:
        dte = None
    if dte is not None:
        dte = dte.where(dte >= 0)            # negative = gift after the current expiration value
        soon = np.where(dte.notna(), (dte <= EXPIRING_SOON_DAYS).astype(float), np.nan)
        out["median_days_gift_to_expiration_lifetime"] = (
            pd.DataFrame({"donor_id": donor, "v": dte.to_numpy()}).groupby("donor_id")["v"].median()
        )
        out[f"pct_gifts_expiring_within_{EXPIRING_SOON_DAYS}d_lifetime"] = _grouped_mean(soon, donor)
        out[f"pct_gifts_expiring_within_{EXPIRING_SOON_DAYS}d_12m"] = _grouped_mean(soon, donor, w12)

    # ---------- school need (rows with a school) ----------
    school_present = d["school_id"].notna() if "school_id" in d.columns else pd.Series(True, index=d.index)
    if "has_school" in d.columns:
        school_present = pd.to_numeric(d["has_school"], errors="coerce").eq(1)
    for col, name in [
        ("school_is_low_income", "low_income"),
        ("school_is_historically_underrepresented_race", "underrep_race"),
        ("school_is_underserved_rural", "underserved_rural"),
    ]:
        if col in d.columns:
            v = _yes_no(d[col]).where(school_present).to_numpy()
            out[f"pct_gifts_{name}_school_lifetime"] = _grouped_mean(v, donor)
            out[f"pct_gifts_{name}_school_12m"] = _grouped_mean(v, donor, w12)

    # ---------- optional donation (tip) behavior ----------
    if "optional_donation_rate" in d.columns:
        rate = pd.to_numeric(d["optional_donation_rate"], errors="coerce")
        tip = np.where(rate.notna(), (rate > 0).astype(float), np.nan)
        t = pd.DataFrame({"donor_id": donor, "tip": tip})
        tg = t.groupby("donor_id")["tip"]
        out["pct_gifts_with_optional_donation_lifetime"] = tg.mean()
        out["pct_gifts_with_optional_donation_12m"] = _grouped_mean(tip, donor, w12)
        out["is_always_tipper_lifetime"] = tg.min()
        out["is_never_tipper_lifetime"] = 1.0 - tg.max()

    # ---------- Save-the-thank-you (STY) opt-in among offered gifts ----------
    if "sty_status" in d.columns:
        sty = d["sty_status"].astype("object")
        offered = sty.isin(["Opted in", "Opted out"])
        v = np.where(offered, sty.eq("Opted in").astype(float), np.nan)
        out["pct_sty_opted_in_lifetime"] = _grouped_mean(v, donor)

    return out


# -----------------------------------------------------------------------------
# 2. Email type mix (Campaign vs Triggered)
# -----------------------------------------------------------------------------
def _email_type_features(email_12m):
    if email_12m is None or email_12m.empty or "email_type" not in email_12m.columns:
        return _empty_frame()
    e = email_12m
    trig = e["email_type"].astype("object").str.lower().eq("triggered")
    camp = e["email_type"].astype("object").str.lower().eq("campaign")
    x = pd.DataFrame({
        "donor_id": e["donor_id"].to_numpy(),
        "sent": e["email_sent_count"].to_numpy(),
        "sent_trig": e["email_sent_count"].where(trig, 0).to_numpy(),
        "sent_camp": e["email_sent_count"].where(camp, 0).to_numpy(),
        "open_camp": e["email_open_count"].where(camp, 0).to_numpy(),
        "click_camp": e["email_click_count"].where(camp, 0).to_numpy(),
        "sent_t": e["email_sent_count"].where(trig, 0).to_numpy(),
        "open_t": e["email_open_count"].where(trig, 0).to_numpy(),
    }).groupby("donor_id").sum()
    out = pd.DataFrame(index=x.index)
    out["pct_triggered_emails_12m"] = x["sent_trig"] / x["sent"].replace(0, np.nan)
    out["campaign_email_open_rate_12m"] = x["open_camp"] / x["sent_camp"].replace(0, np.nan)
    out["campaign_email_click_rate_12m"] = x["click_camp"] / x["sent_camp"].replace(0, np.nan)
    out["triggered_email_open_rate_12m"] = x["open_t"] / x["sent_t"].replace(0, np.nan)
    return out


# -----------------------------------------------------------------------------
# 3. Lifecycle (project + monthly activity combined)
# -----------------------------------------------------------------------------
def _lifecycle_features(dpr_pre_T, md_pre_T, T):
    parts = []
    first_parts = []
    if dpr_pre_T is not None and not dpr_pre_T.empty:
        parts.append(dpr_pre_T[["donor_id", "payment_date"]].rename(columns={"payment_date": "dt"}))
        first_parts.append(dpr_pre_T.groupby("donor_id")["payment_date"].min())
    if md_pre_T is not None and not md_pre_T.empty:
        pay_col = next((c for c in ["charge_date", "monthly_subscription_payment_date"] if c in md_pre_T.columns), None)
        if pay_col is not None:
            mp = md_pre_T[md_pre_T[pay_col] < T]
            parts.append(mp[["donor_id", pay_col]].rename(columns={pay_col: "dt"}))
        first_parts.append(md_pre_T.groupby("donor_id")["monthly_subscription_joined_date"].min())
    if not parts:
        return _empty_frame()

    act = pd.concat(parts, ignore_index=True)
    act = act[act["dt"] < T]
    first_any = pd.concat(first_parts, axis=1).min(axis=1)

    y1, y2 = T - pd.DateOffset(months=12), T - pd.DateOffset(months=24)
    g12 = act[act["dt"] >= y1].groupby("donor_id").size().gt(0)
    g12_24 = act[(act["dt"] >= y2) & (act["dt"] < y1)].groupby("donor_id").size().gt(0)
    pre24 = act[act["dt"] < y2].groupby("donor_id").size().gt(0)

    out = pd.DataFrame(index=pd.Index(first_any.index, name="donor_id"))
    out["tenure_days_any_giving"] = (T - first_any).dt.days
    a12 = g12.reindex(out.index, fill_value=False)
    a12_24 = g12_24.reindex(out.index, fill_value=False)
    ap24 = pre24.reindex(out.index, fill_value=False)
    is_new = first_any.reindex(out.index) >= y1
    out["is_new_donor_12m"] = is_new.astype(int)
    out["is_continuing_donor_12m"] = (a12 & a12_24).astype(int)
    out["is_reactivated_12m"] = (a12 & ~a12_24 & ap24 & ~is_new).astype(int)
    return out


# -----------------------------------------------------------------------------
# 4. Cohort rule as of T (pair with EDIT 3 in dpr_extract_changes.sql)
# -----------------------------------------------------------------------------
def _loyalist_ids_at_T(df_dpr, T, lookback_years=3):
    """Donors meeting the SQL loyalist rule on green giving in [T - lookback, T)."""
    start = T - pd.DateOffset(years=lookback_years)
    pay_date = _as_datetime(df_dpr["payment_date"])
    g = df_dpr[
        (pay_date >= start) & (pay_date < T)
        & df_dpr["is_green_payment"].isin(TRUE_VALS)
    ]
    key = "payment_id" if "payment_id" in g.columns else None
    agg = g.groupby("donor_id").agg(
        n=(key, "nunique") if key else ("payment_amount", "size"),
        amt=("green_payment_amount", "sum"),
    )
    ok = (
        ((agg["n"] >= 8) & (agg["amt"] >= 100)) | ((agg["n"] >= 4) & (agg["amt"] >= 200))
        | ((agg["n"] >= 2) & (agg["amt"] >= 500)) | ((agg["n"] >= 1) & (agg["amt"] >= 1000))
    )
    return pd.Index(agg.index[ok], name="donor_id")

# Applied in build_features (apply_loyalist_rule_at_T=True) to the PROJECT branch only,
# so the monthly branch of the eligible population is unchanged.


# -----------------------------------------------------------------------------
# 5. One-time checks on the inputs these features assume
# -----------------------------------------------------------------------------
def verify_expansion_inputs(df_dpr):
    """Run once on the raw extract (dates may still be strings). Does not modify df_dpr."""
    d = df_dpr
    pay_date = _as_datetime(d["payment_date"])
    if "payment_id" in d.columns:
        print(f"payment_id duplicated: {d['payment_id'].duplicated().mean():.2%}  (should be 0)")
    key = "cart_id" if "cart_id" in d.columns else "donation_n"
    cols = ["donor_id", key] + [c for c in ["referral_source", "payment_type"] if c in d.columns]
    k = d.loc[d[key].notna(), cols]   # column subset, avoids copying the full extract
    for col in ["referral_source", "payment_type"]:
        if col in k.columns:
            multi = k.groupby(["donor_id", key])[col].nunique().gt(1).mean()
            note = "occasion-level shares assume ~0" if col == "referral_source" else "payment type is shared at the gift level, so this is informational"
            print(f"checkouts with >1 distinct {col}: {multi:.2%}  ({note})")
    if "project_expiration_date" in d.columns:
        exp = pd.to_datetime(d["project_expiration_date"], format="%Y-%m-%d", errors="coerce")
        dte = (exp - pay_date.dt.normalize()).dt.days
        print(f"gifts dated after the project's current expiration: {(dte < 0).mean():.2%} of rows with a date")


# -----------------------------------------------------------------------------
# 6. Feature lists for the modeling pipeline (not dropped here, to keep the
#    frame backward compatible)
# -----------------------------------------------------------------------------

# Built from current-state source columns (see SQL review): values in a training frame
# can reflect events after T.
# v1.3.3: these are DROPPED from the output frames in build_features.
SNAPSHOT_DROPPED_FEATURES = [
    "mean_teacher_lifetime_projects_fully_funded",  # current teacher totals
    "mean_teacher_lifetime_donations",
    "latest_teacher_lifetime_projects_fully_funded",
    "pct_projects_fully_funded",                    # current project status
    "latest_project_got_fully_funded",
    "pct_gifts_last_project",                       # current 'last donation' status
    "pct_amount_last_project",
    "latest_donation_is_projects_last",
]
# KEPT in the frames because downstream cohort / opt-out filters use them
# (the campaign cell filters on is_marketing_subscribed). Exclude them from MODEL
# INPUTS via the modeling pipeline's DROP_FEATURES.
SNAPSHOT_FEATURES = [
    "is_major_gift_donor",                          # current Salesforce status
    "is_marketing_subscribed",                      # current subscription / do-not-contact
]

# Superseded by v1.3 occasion-based features (row-level versions are dominated by
# same-cart rows: 55.7% of consecutive-row gaps are 0 days).
SUPERSEDED_FEATURES = {
    "mean_gap_between_gifts_days_last2yr": "giving_day_gap_mean_days_last2yr",
    "cv_gap_between_gifts_days_last2yr": "giving_day_gap_cv_last2yr",
    "days_since_second_to_last_gift": "days_since_prior_giving_day",
}


### 2.14 Account Credit Ledger (v1.3)

Point-in-time credit features and the credit-use label. Supply `df_credit` to `build_features` to enable.

In [17]:
# =============================================================================
# 2.14 ACCOUNT CREDIT LEDGER (v1.3) — point-in-time credit features
#
# Input df_credit: donorid | txn_date | credits | debits | net_change | balance_eod
# Assumed grain: one row per donor per day with credit activity (verify_credit_ledger).
# Source: AccountCredits.csv. This is the only source of account-credit features;
# the old df_dpr.account_credit_balance snapshot (today's balance on every row) is
# no longer used.
#
# Donors with no ledger rows are treated as never having had credit:
# balance / amounts / counts = 0; recency, CV and shares = NaN. (Assumes the ledger
# covers every donor who ever had credit.)
#
# v1.3.2: removed credit_amount_24m (r = 0.97 with gift_amount_nongreen_12m) and
# credit_share_of_giving_capacity_12m (|r| = 0.98 with pct_amount_green_12m); both
# duplicated existing features. Added large-credit features: among net-new donors,
# DAF rate rises with the largest single credit (trend z = 5.7 train / 7.9 OOT).
# =============================================================================

CREDIT_GAP_WINDOW_MONTHS = 36
CREDIT_GAP_MIN_DAYS = 3           # CV needs at least 3 credit days (2 gaps)
LARGE_CREDIT_THRESHOLD = 250      # $250+ single-day credits: ~3x DAF rate in train and OOT
CREDIT_FEATURE_COLUMNS = [
    "credit_balance_at_T",
    "days_since_last_credit",
    f"n_credit_days_{CREDIT_GAP_WINDOW_MONTHS}m",
    f"credit_gap_cv_{CREDIT_GAP_WINDOW_MONTHS}m",
    "credit_amount_lifetime",
    "max_single_credit_lifetime",
    f"n_credits_{LARGE_CREDIT_THRESHOLD}plus_lifetime",
    "largest_credit_share_of_lifetime_credits",
    "days_since_largest_credit",
]


def prepare_credit_ledger(df_credit):
    """Standardize column names/types and sort. Returns a new frame."""
    c = df_credit.copy()
    c.columns = [str(col).strip().lower() for col in c.columns]
    c = c.rename(columns={"donorid": "donor_id"})
    missing = {"donor_id", "txn_date", "credits", "debits", "net_change", "balance_eod"} - set(c.columns)
    if missing:
        raise ValueError(f"AccountCredits is missing columns: {sorted(missing)}")
    if not pd.api.types.is_datetime64_any_dtype(c["txn_date"]):
        parsed = pd.to_datetime(c["txn_date"], format="%m/%d/%y", errors="coerce")
        if parsed.isna().mean() > 0.01:                       # not m/d/yy -> let pandas infer
            parsed = pd.to_datetime(c["txn_date"], errors="coerce")
        c["txn_date"] = parsed
    for col in ["credits", "debits", "net_change"]:
        c[col] = pd.to_numeric(c[col], errors="coerce").fillna(0.0)
    c["balance_eod"] = pd.to_numeric(c["balance_eod"], errors="coerce")
    c["donor_id"] = pd.to_numeric(c["donor_id"], errors="coerce")
    c = c[c["donor_id"].notna() & c["txn_date"].notna()].copy()
    c["donor_id"] = c["donor_id"].astype("int64")
    return c.sort_values(["donor_id", "txn_date"], kind="mergesort").reset_index(drop=True)


def _credit_features(credit_pre_T, dpr_pre_T, T, donor_index):
    """
    Point-in-time account-credit features as of T (ledger rows with txn_date < T).

    A "credit" is one ledger day with credits > 0 (the ledger is one row per donor-day).
    dpr_pre_T is accepted for signature compatibility and is not used.
    """
    out = pd.DataFrame(index=pd.Index(donor_index, name="donor_id"))
    gap_col = f"credit_gap_cv_{CREDIT_GAP_WINDOW_MONTHS}m"
    n_col = f"n_credit_days_{CREDIT_GAP_WINDOW_MONTHS}m"
    large_col = f"n_credits_{LARGE_CREDIT_THRESHOLD}plus_lifetime"

    zero_cols = ["credit_balance_at_T", n_col, "credit_amount_lifetime",
                 "max_single_credit_lifetime", large_col]
    if credit_pre_T is None or credit_pre_T.empty:
        for col in CREDIT_FEATURE_COLUMNS:
            out[col] = 0.0 if col in zero_cols else np.nan
        return out[CREDIT_FEATURE_COLUMNS]

    c = credit_pre_T

    # Balance at T: last end-of-day balance before T
    last = c.groupby("donor_id")["balance_eod"].last()
    out["credit_balance_at_T"] = last.reindex(out.index).fillna(0.0)

    cr = c[c["credits"] > 0]
    by = cr.groupby("donor_id")

    # Recency of the last credit received
    out["days_since_last_credit"] = (T - by["txn_date"].max()).dt.days

    # Regularity of credit arrival, trailing window (+ count fallback)
    cg = cr[cr["txn_date"] >= T - pd.DateOffset(months=CREDIT_GAP_WINDOW_MONTHS)]
    days = cg[["donor_id", "txn_date"]].drop_duplicates()
    out[n_col] = days.groupby("donor_id").size().reindex(out.index).fillna(0.0)
    gaps = pd.DataFrame({
        "donor_id": days["donor_id"].to_numpy(),
        "gap": days.groupby("donor_id")["txn_date"].diff().dt.days.to_numpy(),
    }).dropna()
    g = gaps.groupby("donor_id")["gap"]
    cv = g.std() / g.mean()
    out[gap_col] = cv.where(out[n_col].reindex(cv.index) >= CREDIT_GAP_MIN_DAYS)

    # Lifetime volume and large single credits
    total = by["credits"].sum()
    largest = by["credits"].max()
    out["credit_amount_lifetime"] = total.reindex(out.index).fillna(0.0)
    out["max_single_credit_lifetime"] = largest.reindex(out.index).fillna(0.0)
    out[large_col] = (
        cr[cr["credits"] >= LARGE_CREDIT_THRESHOLD].groupby("donor_id").size()
        .reindex(out.index).fillna(0.0)
    )
    out["largest_credit_share_of_lifetime_credits"] = largest / total.replace(0, np.nan)

    # Days since the largest credit (most recent date if the maximum repeats)
    top = (
        cr.sort_values(["donor_id", "credits", "txn_date"], kind="mergesort")
        .groupby("donor_id").tail(1).set_index("donor_id")["txn_date"]
    )
    out["days_since_largest_credit"] = (T - top).dt.days

    return out[CREDIT_FEATURE_COLUMNS]


def _label_will_use_account_credit(credit, T, horizon_days=90):
    """1 if any ledger debit in [T, T + horizon_days)."""
    if credit is None or credit.empty:
        return pd.Series(dtype=float)
    end = T + pd.Timedelta(days=horizon_days)
    fut = credit[(credit["txn_date"] >= T) & (credit["txn_date"] < end) & (credit["debits"] > 0)]
    return fut.groupby("donor_id").size().gt(0).astype(float)


def verify_credit_ledger(df_credit, df_dpr):
    """Run once on a new ledger extract before building features."""
    raw_n = len(df_credit)
    c = prepare_credit_ledger(df_credit)
    print(f"rows dropped for unparseable donor/date: {1 - len(c) / max(raw_n, 1):.2%}")
    if c.empty:
        print("AccountCredits has no usable rows")
        return
    print(f"ledger rows={len(c):,} donors={c['donor_id'].nunique():,} "
          f"dates {c['txn_date'].min().date()} → {c['txn_date'].max().date()}")
    print(f"rows per donor-day: max={c.groupby(['donor_id', 'txn_date']).size().max()}  (1 = daily grain)")
    print(f"net_change == credits - debits: {((c['credits'] - c['debits'] - c['net_change']).abs() < 0.005).mean():.2%}")
    prev = c.groupby("donor_id")["balance_eod"].shift().fillna(0.0)
    print(f"balance continuity (prev + net == eod): {((prev + c['net_change'] - c['balance_eod']).abs() < 0.005).mean():.2%}")
    print(f"negative balances: {(c['balance_eod'] < -0.005).mean():.2%}")

    # Reconcile ledger debits with non-green (giftcard) DPR payments per donor-day
    d = df_dpr[df_dpr["donor_id"].isin(c["donor_id"].unique())]
    pdt = pd.to_datetime(d["payment_date"]).dt.normalize()
    nongreen = ~d["is_green_payment"].isin(TRUE_VALS)
    dpr_day = (pd.DataFrame({"donor_id": d["donor_id"].to_numpy(), "txn_date": pdt.to_numpy(),
                             "amt": d["payment_amount"].where(nongreen, 0.0).to_numpy()})
               .groupby(["donor_id", "txn_date"])["amt"].sum())
    deb = c[c["debits"] > 0].set_index(["donor_id", "txn_date"])["debits"]
    if d.empty:
        print("no ledger donors found in df_dpr; skipping debit reconciliation")
        return
    lo, hi = pdt.min(), pdt.max()
    deb = deb[(deb.index.get_level_values("txn_date") >= lo) & (deb.index.get_level_values("txn_date") <= hi)]
    joined = deb.rename("debits").to_frame().join(dpr_day.rename("dpr_nongreen"), how="left").fillna(0.0)
    match = ((joined["debits"] - joined["dpr_nongreen"]).abs() < 0.01).mean()
    print(f"ledger debit days matching DPR non-green $ on the same donor-day: {match:.2%}  (n={len(joined):,})")


## 3. Label Group Functions

### 3.1 Core Giving Propensity

In [18]:
def _labels_core_giving_propensity(df_dpr: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    A. Core Giving Propensity.

    Labels:
      - label_will_give_next_30d
      - label_will_give_next_90d
      - label_will_give_next_365d
      - label_num_gifts_next_365d
      - label_num_green_gifts_next_365d
      - label_will_give_2plus_times_next_365d
      - label_time_to_next_gift_days
      - label_will_make_big_event_gift_next_365d
    """
    df = df_dpr.copy()
    df["payment_date"] = pd.to_datetime(df["payment_date"])
    donor_index = df["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    # Future windows
    dpr_30  = _future_window(df, "payment_date", T, 30)
    dpr_90  = _future_window(df, "payment_date", T, 90)
    dpr_365 = _future_window(df, "payment_date", T, 365)

    # 1–3. Any gift in 30/90/365d
    out["label_will_give_next_30d"]  = _any_by_donor(dpr_30).reindex(out.index, fill_value=0.0)
    out["label_will_give_next_90d"]  = _any_by_donor(dpr_90).reindex(out.index, fill_value=0.0)
    out["label_will_give_next_365d"] = _any_by_donor(dpr_365).reindex(out.index, fill_value=0.0)
    
    # 1–3.b. No gift in 30/90/365d
    out["label_will_not_give_next_30d"]  = 1.0 - out["label_will_give_next_30d"]
    out["label_will_not_give_next_90d"]  = 1.0 - out["label_will_give_next_90d"]
    out["label_will_not_give_next_365d"] = 1.0 - out["label_will_give_next_365d"]

    # 4. Num_gifts_next_365d
    out["label_num_gifts_next_365d"] = _count_by_donor(dpr_365)

    # 5. Num_green_gifts_next_365d
    if "is_green_payment" in df.columns:
        green_mask = dpr_365["is_green_payment"].isin(["t", "T", True, 1, "1"])
        green_365 = dpr_365[green_mask]
    elif "green_payment_amount" in df.columns:
        green_365 = dpr_365[dpr_365["green_payment_amount"].fillna(0) > 0]
    else:
        green_365 = dpr_365.iloc[0:0]
    out["label_num_green_gifts_next_365d"] = _count_by_donor(green_365)

    # 6. Will_give_2plus_times_next_365d
    # v1.3.3: 2+ checkout occasions (not 2+ payment rows; one cart can fan out to many
    # project rows). label_num_gifts_* intentionally still count project gifts.
    n_occ_365 = (
        pd.DataFrame({"donor_id": dpr_365["donor_id"].to_numpy(),
                      "occ": _occasion_key(dpr_365).to_numpy()})
        .drop_duplicates()
        .groupby("donor_id").size()
    )
    out["label_will_give_2plus_times_next_365d"] = (
        n_occ_365.reindex(out.index) >= 2
    ).astype(float)

    # 7. Time_to_next_gift_days (v1.3.5: within the 365d window; none -> censored)
    future_gifts = (
        dpr_365
        .sort_values(["donor_id", "payment_date"])
        .groupby("donor_id")["payment_date"]
        .min()
    )
    time_to_next = (future_gifts - T).dt.days.astype(float)
    out["label_time_to_next_gift_days"] = time_to_next

    # 8. Will_make_big_event_gift_next_365d
    if "payment_on_big_event" in df.columns:
        big_event_365 = dpr_365[dpr_365["payment_on_big_event"] == 1]
        out["label_will_make_big_event_gift_next_365d"] = _any_by_donor(big_event_365)
    else:
        out["label_will_make_big_event_gift_next_365d"] = np.nan

    return out


# ---------------------------------------------------------------------
# B. Revenue & Value (8 labels)
# ---------------------------------------------------------------------


### 3.2 Revenue & Value

In [19]:
def _labels_revenue_and_value(df_dpr: pd.DataFrame, df_monthly: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    B. Revenue & Value.

    Labels:
      - label_green_revenue_next_365d
      - label_project_revenue_next_365d
      - label_monthly_revenue_next_365d
      - label_grand_revenue_next_365d
      - label_max_single_gift_next_365d
      - label_revenue_growth_ratio_next_year
      - label_will_cross_midlevel_threshold_next_12m
      - label_will_make_major_gift_next_24m
    """
    dpr = df_dpr.copy()
    dpr["payment_date"] = pd.to_datetime(dpr["payment_date"])

    monthly = df_monthly.copy() if df_monthly is not None else pd.DataFrame()
    if not monthly.empty and "charge_date" in monthly.columns:
        monthly["charge_date"] = pd.to_datetime(monthly["charge_date"])

    # v1.3.4: index on project AND monthly donors. Indexing on df_dpr alone dropped
    # monthly-only donors, so their monthly/grand revenue labels were imputed to 0.
    donor_ids = [dpr["donor_id"]]
    if not monthly.empty and "donor_id" in monthly.columns:
        donor_ids.append(monthly["donor_id"])
    donor_index = pd.Index(
        pd.unique(pd.concat(donor_ids, ignore_index=True)), name="donor_id"
    ).sort_values()
    out = pd.DataFrame(index=donor_index)

    # Future windows
    dpr_365 = _future_window(dpr, "payment_date", T, 365)
    dpr_24m = _future_window(dpr, "payment_date", T, 365 * 2)
    if not monthly.empty and "charge_date" in monthly.columns:
        monthly_365 = _future_window(monthly, "charge_date", T, 365)
    else:
        monthly_365 = pd.DataFrame()

    # 9. Green_revenue_next_365d
    if "green_payment_amount" in dpr.columns:
        green_365 = dpr_365.copy()
        green_365["green_payment_amount"] = green_365["green_payment_amount"].fillna(0.0)
        out["label_green_revenue_next_365d"] = _sum_by_donor(green_365, "green_payment_amount")
    elif "is_green_payment" in dpr.columns:
        green_mask = dpr_365["is_green_payment"].isin(["t", "T", True, 1, "1"])
        green_365 = dpr_365[green_mask]
        out["label_green_revenue_next_365d"] = _sum_by_donor(green_365, "payment_amount")
    else:
        out["label_green_revenue_next_365d"] = np.nan

    # 10. Project_revenue_next_365d (project giving dollars in df_dpr)
    out["label_project_revenue_next_365d"] = _sum_by_donor(dpr_365, "payment_amount")

    # 11. Monthly_revenue_next_365d (from monthly charges)
    if not monthly_365.empty and "monthly_subscription_payment_amount" in monthly_365.columns:
        out["label_monthly_revenue_next_365d"] = _sum_by_donor(
            monthly_365, "monthly_subscription_payment_amount"
        )
    else:
        out["label_monthly_revenue_next_365d"] = np.nan

    # 12. Grand_revenue_next_365d = project + monthly
    out["label_grand_revenue_next_365d"] = (
        out["label_project_revenue_next_365d"].fillna(0.0)
        + out["label_monthly_revenue_next_365d"].fillna(0.0)
    )

    # 13. Max_single_gift_next_365d
    out["label_max_single_gift_next_365d"] = _max_by_donor(dpr_365, "payment_amount")

    # 14. Revenue_growth_ratio_next_year = future 12m / prior 12m
    past_365 = dpr[
        (dpr["payment_date"] >= (T - pd.Timedelta(days=365)))
        & (dpr["payment_date"] < T)
    ]
    past_sum = _sum_by_donor(past_365, "payment_amount")
    future_sum = _sum_by_donor(dpr_365, "payment_amount")
    denom = past_sum.replace({0.0: np.nan})
    out["label_revenue_growth_ratio_next_year"] = future_sum / denom

    # 15. Will_cross_midlevel_threshold_next_12m (using grand future 12m)
    out["label_will_cross_midlevel_threshold_next_12m"] = (
        out["label_grand_revenue_next_365d"] >= MIDLEVEL_THRESHOLD
    ).astype(float)

    # 16. Will_cross_tiptop_threshold_next_12m (using grand future 12m)
    out["label_will_cross_tiptop_threshold_next_12m"] = (
        out["label_grand_revenue_next_365d"] >= TIPTOP_THRESHOLD
    ).astype(float)

    # 17. Will_make_major_gift_next_24m (v1.3.3): any single payment >= TIPTOP_THRESHOLD
    # in [T, T + 730d). NaN for every donor when that window runs past the data (e.g. the
    # OOT frame); _apply_label_imputation then leaves it NaN instead of imputing 0.
    observed_end = dpr["payment_date"].max()
    if pd.notna(observed_end) and T + pd.Timedelta(days=365 * 2) <= observed_end + pd.Timedelta(days=1):
        major_24m = dpr_24m[dpr_24m["payment_amount"] >= TIPTOP_THRESHOLD]
        out["label_will_make_major_gift_next_24m"] = (
            _any_by_donor(major_24m).reindex(out.index).fillna(0.0)
        )
    else:
        out["label_will_make_major_gift_next_24m"] = np.nan

    return out


# ---------------------------------------------------------------------
# C. Monthly Program (6 labels)
# ---------------------------------------------------------------------


### 3.3 Monthly Program

In [20]:
def _labels_monthly_program(df_monthly: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    C. Monthly Program.

    Labels:
      - label_will_start_monthly_next_90d
      - label_will_start_monthly_next_365d
      - label_will_churn_monthly_next_6m
      - label_monthly_amt_next_6m
      - label_will_upgrade_monthly_next_6m
      - label_will_downgrade_monthly_next_6m
    """
    if df_monthly is None or len(df_monthly) == 0:
        return pd.DataFrame(index=pd.Index([], name="donor_id"))

    m = df_monthly.copy()
    if "monthly_subscription_joined_date" in m.columns:
        m["monthly_subscription_joined_date"] = pd.to_datetime(
            m["monthly_subscription_joined_date"]
        )
    if "monthly_subscription_retired_date" in m.columns:
        m["monthly_subscription_retired_date"] = pd.to_datetime(
            m["monthly_subscription_retired_date"]
        )
    if "charge_date" in m.columns:
        m["charge_date"] = pd.to_datetime(m["charge_date"])

    donor_index = m["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    # Active at T
    if {"monthly_subscription_joined_date", "monthly_subscription_retired_date"} <= set(m.columns):
        active_at_T = (
            (m["monthly_subscription_joined_date"] <= T)
            & (
                m["monthly_subscription_retired_date"].isna()
                | (m["monthly_subscription_retired_date"] > T)
            )
        )
    else:
        # If subscription dates are missing, we cannot determine activity; return NaNs.
        out["label_will_start_monthly_next_90d"] = np.nan
        out["label_will_start_monthly_next_365d"] = np.nan
        out["label_will_churn_monthly_next_6m"] = np.nan
        out["label_monthly_amt_next_6m"] = np.nan
        out["label_will_upgrade_monthly_next_6m"] = np.nan
        out["label_will_downgrade_monthly_next_6m"] = np.nan
        return out

    # 17–18. Will_start_monthly_next_90d / 365d among non-monthly donors at T
    non_active_at_T = ~active_at_T

    join_90 = (
        (m["monthly_subscription_joined_date"] >= T)
        & (m["monthly_subscription_joined_date"] < T + pd.Timedelta(days=90))
    )
    join_365 = (
        (m["monthly_subscription_joined_date"] >= T)
        & (m["monthly_subscription_joined_date"] < T + pd.Timedelta(days=365))
    )

    will_start_90 = m[non_active_at_T & join_90]
    will_start_365 = m[non_active_at_T & join_365]

    out["label_will_start_monthly_next_90d"] = _any_by_donor(will_start_90)
    out["label_will_start_monthly_next_365d"] = _any_by_donor(will_start_365)

    # 19. Will_churn_monthly_next_6m (active at T and retire in next 180d)
    retire_6m = (
        (m["monthly_subscription_retired_date"] >= T)
        & (m["monthly_subscription_retired_date"] < T + pd.Timedelta(days=180))
    )
    churn_candidates = m[active_at_T & retire_6m]
    out["label_will_churn_monthly_next_6m"] = _any_by_donor(churn_candidates)

    # 20. Monthly_amt_next_6m = total subscription charges in 6m
    if "charge_date" in m.columns and "monthly_subscription_payment_amount" in m.columns:
        charges_6m = _future_window(m, "charge_date", T, 180)
        out["label_monthly_amt_next_6m"] = _sum_by_donor(
            charges_6m, "monthly_subscription_payment_amount"
        )
    else:
        out["label_monthly_amt_next_6m"] = np.nan

    # 21–22. Upgrade / downgrade within 6m
    if "charge_date" in m.columns and "monthly_subscription_payment_amount" in m.columns:
        past_6m = m[
            (m["charge_date"] >= (T - pd.Timedelta(days=180)))
            & (m["charge_date"] < T)
        ]
        future_6m = charges_6m

        past_med = past_6m.groupby("donor_id")[
            "monthly_subscription_payment_amount"
        ].median()
        fut_med = future_6m.groupby("donor_id")[
            "monthly_subscription_payment_amount"
        ].median()

        diff = fut_med - past_med.reindex(fut_med.index)
        out["label_will_upgrade_monthly_next_6m"] = (diff > 0).astype(float)
        out["label_will_downgrade_monthly_next_6m"] = (diff < 0).astype(float)
    else:
        out["label_will_upgrade_monthly_next_6m"] = np.nan
        out["label_will_downgrade_monthly_next_6m"] = np.nan

    return out


### 3.4 Midlevel / Major / DAF

In [21]:
def _labels_mid_major_daf(df_dpr: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    D. Midlevel / Major / DAF.

    Labels:
      - label_daf_revenue_next_365d
      - label_will_make_daf_gift_next_365d
      - label_will_shift_to_daf_heavy_next_12m
      - label_will_be_midlevel_grand_next_12m
      - label_will_make_large_single_gift_next_365d
    """
    dpr = df_dpr.copy()
    dpr["payment_date"] = pd.to_datetime(dpr["payment_date"])

    # Ensure daf_payment column exists
    if "daf_payment" not in dpr.columns:
        dpr["daf_payment"] = 0

    donor_index = dpr["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    dpr_365 = _future_window(dpr, "payment_date", T, 365)
    dpr_12m = dpr_365
    dpr_24m = _future_window(dpr, "payment_date", T, 365 * 2)

    # 23. DAF_revenue_next_365d
    daf_365 = dpr_365[dpr_365["daf_payment"] == 1]
    out["label_daf_revenue_next_365d"] = _sum_by_donor(daf_365, "payment_amount")

    # 24. Will_make_DAF_gift_next_365d
    out["label_will_make_daf_gift_next_365d"] = _any_by_donor(daf_365)

    # 25. Will_shift_to_DAF_heavy_next_12m (>=50% future dollars via DAF)
    future_sum = _sum_by_donor(dpr_12m, "payment_amount")
    daf_sum = _sum_by_donor(daf_365, "payment_amount")
    share_daf = daf_sum / future_sum.replace({0.0: np.nan})
    out["label_will_shift_to_daf_heavy_next_12m"] = (share_daf >= 0.5).astype(float)

    # v1.3: narrow DAF definition (DONOR_ADVISED_FUND / CHARIOT_DAF only), when available.
    # daf_payment also flags IRA, INDIVIDUAL, FOUNDATION and ACH checks.
    if "is_daf_vehicle" in dpr_365.columns:
        dafv_365 = dpr_365[pd.to_numeric(dpr_365["is_daf_vehicle"], errors="coerce") == 1]
        out["label_daf_vehicle_revenue_next_365d"] = _sum_by_donor(dafv_365, "payment_amount")
        out["label_will_make_daf_vehicle_gift_next_365d"] = _any_by_donor(dafv_365)

    # 26. Will_be_midlevel_grand_next_12m
    # Project-only placeholder; v1.3.4 overwrites it in _build_labels with grand revenue
    # (project + monthly) from _labels_revenue_and_value.
    out["label_will_be_midlevel_grand_next_12m"] = (
        future_sum >= MIDLEVEL_THRESHOLD
    ).astype(float)

    # 27. Will_make_large_single_gift_next_365d (>= configurable $X)
    large_365 = dpr_365[dpr_365["payment_amount"] >= LARGE_SINGLE_GIFT_THRESHOLD]
    out["label_will_make_large_single_gift_next_365d"] = _any_by_donor(large_365)

    return out


# ---------------------------------------------------------------------
# E. Email Engagement (5 labels)
# ---------------------------------------------------------------------


### 3.5 Email Engagement

In [22]:
def _labels_email_engagement(df_email: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    E. Email Engagement.

    Labels:
      - label_email_open_rate_next_90d
      - label_email_click_rate_next_90d
      - label_will_open_next_campaign_email
      - label_will_click_next_campaign_email
      - label_will_unsubscribe_next_90d

    This implementation assumes df_email is monthly-aggregated email activity
    with columns like:
      - donor_id
      - email_sent_month (or send_datetime)
      - email_sent_count
      - email_open_count
      - email_click_count

    Campaign-level and unsubscribe labels are set to NaN unless the relevant
    event-level columns are present.
    """
    if df_email is None or len(df_email) == 0:
        return pd.DataFrame(index=pd.Index([], name="donor_id"))

    email = df_email.copy()
    date_col = "send_datetime" if "send_datetime" in email.columns else "email_sent_month"
    email[date_col] = pd.to_datetime(email[date_col])

    donor_index = email["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    email_90 = _future_window(email, date_col, T, 90)

    # 28–29. Open/click rate in next 90d: open_count / sent_count
    if {"email_sent_count", "email_open_count", "email_click_count"} <= set(email_90.columns):
        sent = email_90.groupby("donor_id")["email_sent_count"].sum()
        opened = email_90.groupby("donor_id")["email_open_count"].sum()
        clicked = email_90.groupby("donor_id")["email_click_count"].sum()

        denom = sent.replace({0.0: np.nan})
        out["label_email_open_rate_next_90d"] = (opened / denom).astype(float)
        out["label_email_click_rate_next_90d"] = (clicked / denom).astype(float)
    else:
        out["label_email_open_rate_next_90d"] = np.nan
        out["label_email_click_rate_next_90d"] = np.nan

    # 30-31 (v1.3.1). With monthly data (email_type = Campaign/Triggered), "next campaign
    # email" is approximated by the first month on/after T with campaign sends:
    # opened (clicked) >= 1 campaign email that month.
    if {"email_type", "email_sent_count", "email_open_count", "email_click_count"} <= set(email.columns) \
            and not {"is_campaign_email", "opened_flag"} <= set(email.columns):
        # v1.3.5: first campaign month within the 90d window (was open-ended)
        camp = email_90[
            email_90["email_type"].astype(str).str.lower().eq("campaign")
            & (email_90["email_sent_count"] > 0)
        ]
        first_month = camp.groupby("donor_id")[date_col].transform("min")
        fm = camp[camp[date_col] == first_month].groupby("donor_id")[["email_open_count", "email_click_count"]].sum()
        out["label_will_open_next_campaign_email"] = (fm["email_open_count"] > 0).astype(float)
        out["label_will_click_next_campaign_email"] = (fm["email_click_count"] > 0).astype(float)
    elif {"is_campaign_email", "opened_flag"} <= set(email.columns):
        campaign = email[email["is_campaign_email"] == 1]  # event-level variant
        future_campaign = campaign[campaign[date_col] > T]
        first_campaign = (
            future_campaign
            .sort_values(["donor_id", date_col])
            .groupby("donor_id")
            .head(1)
        )
        opened = first_campaign.set_index("donor_id")["opened_flag"].astype(float)
        out["label_will_open_next_campaign_email"] = opened
    else:
        out["label_will_open_next_campaign_email"] = np.nan

    # 31. Will_click_next_campaign_email
    if "label_will_click_next_campaign_email" in out.columns:
        pass  # set above from monthly campaign rows
    elif {"is_campaign_email", "clicked_flag"} <= set(email.columns):
        campaign = email[email["is_campaign_email"] == 1]
        future_campaign = campaign[campaign[date_col] > T]
        first_campaign = (
            future_campaign
            .sort_values(["donor_id", date_col])
            .groupby("donor_id")
            .head(1)
        )
        clicked = first_campaign.set_index("donor_id")["clicked_flag"].astype(float)
        out["label_will_click_next_campaign_email"] = clicked
    else:
        out["label_will_click_next_campaign_email"] = np.nan

    # 32. Will_unsubscribe_next_90d
    if "event_type" in email_90.columns:
        unsub_90 = email_90[email_90["event_type"] == "unsubscribe"]
        out["label_will_unsubscribe_next_90d"] = _any_by_donor(unsub_90)
    elif "unsubscribe_datetime" in email.columns:
        email["unsubscribe_datetime"] = pd.to_datetime(email["unsubscribe_datetime"])
        unsub_90 = email[
            (email["unsubscribe_datetime"] >= T)
            & (email["unsubscribe_datetime"] < T + pd.Timedelta(days=90))
        ]
        out["label_will_unsubscribe_next_90d"] = _any_by_donor(unsub_90)
    else:
        pass  # v1.3.1: no unsubscribe data in the extract -> label not created (was NaN -> imputed all-0)

    return out


# ---------------------------------------------------------------------
# F. Site Behavior & Activity (5 labels)
# ---------------------------------------------------------------------


### 3.6 Site Behavior & Activity

In [23]:
def _labels_site_behavior(df_site: pd.DataFrame, df_dpr: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    F. Site Behavior & Activity.

    Labels:
      - label_will_visit_site_next_30d
      - label_num_sessions_next_90d
      - label_project_page_views_next_90d
      - label_will_view_local_projects_next_90d
      - label_will_login_next_30d

    This implementation expects df_site to contain at least:
      - donor_id
      - activity_date
      - optionally: session_id, project_page_visits_day, is_login_event

    The "local project view" label is currently left as NaN unless project-
    level pageview data with location is available.
    """
    if df_site is None or len(df_site) == 0:
        return pd.DataFrame(index=pd.Index([], name="donor_id"))

    site = df_site.copy()
    if "activity_date" not in site.columns:
        return pd.DataFrame(index=pd.Index([], name="donor_id"))

    site["activity_date"] = pd.to_datetime(site["activity_date"])
    donor_index = site["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    site_30 = _future_window(site, "activity_date", T, 30)
    site_90 = _future_window(site, "activity_date", T, 90)

    # 33. Will_visit_site_next_30d
    out["label_will_visit_site_next_30d"] = _any_by_donor(site_30)

    # 34. Num_sessions_next_90d
    if "sessions_day" in site_90.columns:
        # v1.3.1: df_site is one row per donor-day; v1.2 counted active days as sessions
        out["label_num_sessions_next_90d"] = site_90.groupby("donor_id")["sessions_day"].sum().astype(float)
    elif "session_id" in site_90.columns:
        sessions_90 = site_90.drop_duplicates(["donor_id", "session_id"])
        out["label_num_sessions_next_90d"] = _count_by_donor(sessions_90)
    else:
        sessions_90 = site_90.drop_duplicates(["donor_id", "activity_date"])
        out["label_num_sessions_next_90d"] = _count_by_donor(sessions_90)

    # 35. Project_page_views_next_90d
    if "project_page_visits_day" in site_90.columns:
        pv = site_90.groupby("donor_id")[
            "project_page_visits_day"
        ].sum().astype(float)
        out["label_project_page_views_next_90d"] = pv
    else:
        out["label_project_page_views_next_90d"] = np.nan

    # 36. Will_view_local_projects_next_90d
    # out["label_will_view_local_projects_next_90d"] = np.nan

    # 37. Will_login_next_30d
    if "is_login_event" in site_30.columns:
        login_30 = site_30[site_30["is_login_event"] == 1]
        out["label_will_login_next_30d"] = _any_by_donor(login_30)
    else:
        pass  # v1.3.1: no login events in the extract -> label not created (was NaN -> imputed all-0)

    return out


# ---------------------------------------------------------------------
# G. Product & Project Preference (6 labels)
# ---------------------------------------------------------------------


### 3.7 Product & Project Preference

In [24]:
def _labels_product_and_project_preference(df_dpr: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    G. Product & Project Preference.

    Labels:
      - label_next_gift_category
      - label_next_gift_grade_band
      - label_will_give_local_next_365d
      - label_pct_future_dollars_to_top_category_next_365d
      - label_will_give_to_teacher_previously_supported_next_365d
      - label_will_give_to_new_schools_next_365d
    """
    dpr = df_dpr.copy()
    dpr["payment_date"] = pd.to_datetime(dpr["payment_date"])
    donor_index = dpr["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    future_365 = _future_window(dpr, "payment_date", T, 365)
    future_all = future_365   # v1.3.5: next-gift labels use the first gift within 365d (was open-ended)
    history = dpr[dpr["payment_date"] < T]

    # 38. Next_gift_category: category of first future gift
    if "project_category" in dpr.columns:
        first_future = (
            future_all
            .sort_values(["donor_id", "payment_date"])
            .groupby("donor_id")
            .head(1)
        )
        out["label_next_gift_category"] = first_future.set_index("donor_id")[
            "project_category"
        ]
    else:
        out["label_next_gift_category"] = np.nan

    # 39. Next_gift_grade_band
    if "project_grade" in dpr.columns:
        if "project_category" not in dpr.columns:
            first_future = (
                future_all
                .sort_values(["donor_id", "payment_date"])
                .groupby("donor_id")
                .head(1)
            )
        out["label_next_gift_grade_band"] = first_future.set_index("donor_id")[
            "project_grade"
        ]
    else:
        out["label_next_gift_grade_band"] = np.nan

    # 40. Will_give_local_next_365d (distance_mi <= local_radius)
    local_radius_mi = 25.0
    if "distance_mi" in future_365.columns:
        local_365 = future_365[future_365["distance_mi"] <= local_radius_mi]
        out["label_will_give_local_next_365d"] = _any_by_donor(local_365)
    else:
        out["label_will_give_local_next_365d"] = np.nan

    # 41. Pct_future_dollars_to_top_category_next_365d
    if {"project_category", "payment_amount"} <= set(future_365.columns):
        cat_amount = future_365.groupby(["donor_id", "project_category"])[
            "payment_amount"
        ].sum()
        total_amount = future_365.groupby("donor_id")["payment_amount"].sum()
        if not cat_amount.empty:
            top_cat = cat_amount.groupby("donor_id").max()
            pct_top = top_cat / total_amount.replace({0.0: np.nan})
            out["label_pct_future_dollars_to_top_category_next_365d"] = (
                pct_top.astype(float)
            )
        else:
            out["label_pct_future_dollars_to_top_category_next_365d"] = np.nan
    else:
        out["label_pct_future_dollars_to_top_category_next_365d"] = np.nan

    # 42. Will_give_to_teacher_previously_supported_next_365d
    if "teacher_id" in dpr.columns:
        prior_teachers = (
            history.groupby("donor_id")["teacher_id"].unique().to_dict()
        )
        future_teachers = (
            future_365.groupby("donor_id")["teacher_id"].unique().to_dict()
        )

        def _teacher_loyalty(d_id):
            hist = set(prior_teachers.get(d_id, []))
            fut = set(future_teachers.get(d_id, []))
            return float(len(hist & fut) > 0)

        out["label_will_give_to_teacher_previously_supported_next_365d"] = (
            out.index.to_series().map(_teacher_loyalty)
        )
    else:
        out["label_will_give_to_teacher_previously_supported_next_365d"] = np.nan

    # 43. Will_give_to_new_schools_next_365d
    if "school_id" in dpr.columns:
        prior_schools = (
            history.groupby("donor_id")["school_id"].unique().to_dict()
        )
        future_schools = (
            future_365.groupby("donor_id")["school_id"].unique().to_dict()
        )

        def _new_school(d_id):
            hist = set(prior_schools.get(d_id, []))
            fut = set(future_schools.get(d_id, []))
            return float(len(fut - hist) > 0)

        out["label_will_give_to_new_schools_next_365d"] = (
            out.index.to_series().map(_new_school)
        )
    else:
        out["label_will_give_to_new_schools_next_365d"] = np.nan

    return out


# ---------------------------------------------------------------------
# H. Seasonality & Events (4 labels)
# ---------------------------------------------------------------------


### 3.8 Seasonality & Events

In [25]:
def _labels_seasonality_and_events(df_dpr: pd.DataFrame, T: pd.Timestamp) -> pd.DataFrame:
    """
    H. Seasonality & Events.

    Labels:
      - label_will_give_in_back_to_school_season
      - label_will_give_during_eoy
      - label_will_be_event_responsive_next_12m
      - label_quarter_of_max_future_giving_next_12m
    """
    dpr = df_dpr.copy()
    dpr["payment_date"] = pd.to_datetime(dpr["payment_date"])
    donor_index = dpr["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    dpr_12m = _future_window(dpr, "payment_date", T, 365)
    if len(dpr_12m) == 0:
        out["label_will_give_in_back_to_school_season"] = np.nan
        out["label_will_give_during_eoy"] = np.nan
        out["label_will_be_event_responsive_next_12m"] = np.nan
        out["label_quarter_of_max_future_giving_next_12m"] = np.nan
        return out

    dpr_12m = dpr_12m.copy()
    dpr_12m["month"] = dpr_12m["payment_date"].dt.month

    # 44. Will_give_in_back_to_school_season (Aug–Sep)
    bts = dpr_12m[dpr_12m["month"].isin([8, 9])]
    out["label_will_give_in_back_to_school_season"] = _any_by_donor(bts)

    # 45. Will_give_during_EOY (Dec 24–31)
    eoy = dpr_12m[
        (dpr_12m["payment_date"].dt.month == 12)
        & (dpr_12m["payment_date"].dt.day >= 24)
    ]
    out["label_will_give_during_eoy"] = _any_by_donor(eoy)

    # 46. Will_be_event_responsive_next_12m (>= N gifts on defined events)
    N = 2
    if "payment_on_big_event" in dpr_12m.columns:
        events = dpr_12m[dpr_12m["payment_on_big_event"] == 1]
        counts = _count_by_donor(events)
        out["label_will_be_event_responsive_next_12m"] = (counts >= N).astype(float)
    else:
        out["label_will_be_event_responsive_next_12m"] = np.nan

    # 47. Quarter_of_max_future_giving_next_12m (rolling quarters from T)
    if "payment_amount" in dpr_12m.columns:
        dpr_12m["days_since_T"] = (dpr_12m["payment_date"] - T).dt.days
        dpr_12m["quarter"] = pd.cut(
            dpr_12m["days_since_T"],
            bins=[-1, 89, 179, 269, 365],
            labels=["Q1", "Q2", "Q3", "Q4"],
        )
        amount_by_q = dpr_12m.groupby(["donor_id", "quarter"])["payment_amount"].sum()
        if not amount_by_q.empty:
            q_sum = amount_by_q.unstack(fill_value=0.0)
            quarter_max = q_sum.idxmax(axis=1)
            out["label_quarter_of_max_future_giving_next_12m"] = quarter_max
        else:
            out["label_quarter_of_max_future_giving_next_12m"] = np.nan
    else:
        out["label_quarter_of_max_future_giving_next_12m"] = np.nan

    return out


# ---------------------------------------------------------------------
# I. Operational / Mechanism Behavior (3 labels)
# ---------------------------------------------------------------------


### 3.9 Operational / Mechanism Behavior

In [26]:
def _labels_operational_mechanism(df_dpr: pd.DataFrame, T: pd.Timestamp, df_credit=None) -> pd.DataFrame:
    """
    I. Operational / Mechanism Behavior.

    Labels:
      - label_will_use_account_credit_next_90d
      - label_optional_donation_rate_on_next_gift
      - label_will_make_gift_card_purchase_next_12m
    """
    dpr = df_dpr.copy()
    dpr["payment_date"] = pd.to_datetime(dpr["payment_date"])
    donor_index = dpr["donor_id"].drop_duplicates().sort_values()
    out = pd.DataFrame(index=donor_index)

    dpr_90 = _future_window(dpr, "payment_date", T, 90)
    dpr_12m = _future_window(dpr, "payment_date", T, 365)

    # 48. Will_use_account_credit_next_90d
    # v1.3: any AccountCredits ledger debit in [T, T+90d). Before v1.3 this label read
    # account_credit_amount, which is not in the extract, so it was all-NaN -> imputed 0.
    if df_credit is not None and not df_credit.empty:
        out["label_will_use_account_credit_next_90d"] = _label_will_use_account_credit(df_credit, T, 90)
    else:
        pass  # no AccountCredits ledger supplied -> label not created

    # 49. Optional_donation_rate_on_next_gift (v1.3.5: first gift within 365d)
    next_future = (
        dpr_12m
        .sort_values(["donor_id", "payment_date"])
        .groupby("donor_id")
        .head(1)
    )
    if "optional_donation_rate" in next_future.columns:
        rate = next_future.set_index("donor_id")[
            "optional_donation_rate"
        ].astype(float)
        out["label_optional_donation_rate_on_next_gift"] = rate
    else:
        out["label_optional_donation_rate_on_next_gift"] = np.nan

    # 50. Will_make_gift_card_purchase_next_12m
    if "gift_card_purchase" in dpr_12m.columns:
        gc_12m = dpr_12m[dpr_12m["gift_card_purchase"] == 1]
        out["label_will_make_gift_card_purchase_next_12m"] = _any_by_donor(gc_12m)
    else:
        out["label_will_make_gift_card_purchase_next_12m"] = np.nan

    return out


## 4. Construction Functions

### 4.1 Build Labels

In [27]:
def _join_group(frame, part, group, registry):
    """Left-join a feature/label group and record which group each column came from."""
    registry.update({c: group for c in part.columns})
    return frame.join(part, how="left")


def _build_labels(
    df_dpr: pd.DataFrame,
    df_monthly: pd.DataFrame,
    df_share: pd.DataFrame,
    df_email: pd.DataFrame,
    df_site: pd.DataFrame,
    T,
    H=None,
    donor_index=None,
    df_credit=None,
) -> pd.DataFrame:
    """
    Build future-focused labels for multiple horizons and behaviors.

    If donor_index is provided, it will be used as the row universe for labels
    (e.g., the same as the feature frame index). Otherwise, donors present in
    df_dpr are used.
    """
    T = pd.to_datetime(T)
    _validate_dpr_for_labels(df_dpr)

    if donor_index is None:
        donor_index = (
            df_dpr["donor_id"]
            .drop_duplicates()
            .sort_values()
        )
    else:
        # Ensure it's a clean, sorted Index
        donor_index = (
            pd.Index(donor_index)
            .drop_duplicates()
            .sort_values()
        )

    labels = pd.DataFrame(index=donor_index)
    label_groups = {}   # v1.3.5: label -> group, for the frame manifest

    # A. Core Giving Propensity
    labels = _join_group(labels, _labels_core_giving_propensity(df_dpr, T), "core", label_groups)

    # B. Revenue & Value
    labels = _join_group(labels, _labels_revenue_and_value(df_dpr, df_monthly, T), "revenue", label_groups)

    # C. Monthly Program
    labels = _join_group(labels, _labels_monthly_program(df_monthly, T), "monthly_program", label_groups)

    # D. Midlevel / Major / DAF
    labels = _join_group(labels, _labels_mid_major_daf(df_dpr, T), "mid_major", label_groups)

    # v1.3.4: "grand" = project + monthly. Overwrite in place (keeps column position).
    labels["label_will_be_midlevel_grand_next_12m"] = (
        labels["label_grand_revenue_next_365d"].fillna(0.0) >= MIDLEVEL_THRESHOLD
    ).astype(float)

    # E. Email Engagement
    labels = _join_group(labels, _labels_email_engagement(df_email, T), "email", label_groups)

    # F. Site Behavior & Activity
    labels = _join_group(labels, _labels_site_behavior(df_site, df_dpr, T), "site", label_groups)

    # G. Product & Project Preference
    labels = _join_group(labels, _labels_product_and_project_preference(df_dpr, T), "preference", label_groups)

    # H. Seasonality & Events
    labels = _join_group(labels, _labels_seasonality_and_events(df_dpr, T), "seasonality", label_groups)

    # I. Operational / Mechanism Behavior
    labels = _join_group(labels, _labels_operational_mechanism(df_dpr, T, df_credit=df_credit), "operational", label_groups)

    labels.attrs["label_groups"] = label_groups
    return labels

### 4.2 Impute Labels

In [28]:
def _apply_label_imputation(features: pd.DataFrame) -> pd.DataFrame:
    """
    Apply label-specific imputations in place, returning the modified DataFrame.

    This expects label columns to be prefixed with 'label_' and uses the
    imputation rules agreed in the label design (binary -> 0, counts -> 0,
    amounts -> 0, rates -> 0, ratio -> 1.0, time-to-event -> sentinel,
    categorical -> 'NONE').
    """
    if features is None or features.empty:
        return features

    features = features.copy()

    # Binary 0/1 labels
    binary_labels = [
        "label_will_give_next_30d",
        "label_will_give_next_90d",
        "label_will_give_next_365d",
        "label_will_not_give_next_30d",
        "label_will_not_give_next_90d",
        "label_will_not_give_next_365d",
        "label_will_give_2plus_times_next_365d",
        "label_will_make_big_event_gift_next_365d",
        "label_will_cross_midlevel_threshold_next_12m",
        "label_will_start_monthly_next_90d",
        "label_will_start_monthly_next_365d",
        "label_will_churn_monthly_next_6m",
        "label_will_upgrade_monthly_next_6m",
        "label_will_downgrade_monthly_next_6m",
        "label_will_make_daf_gift_next_365d",
        "label_will_make_daf_vehicle_gift_next_365d",
        "label_will_shift_to_daf_heavy_next_12m",
        "label_will_be_midlevel_grand_next_12m",
        "label_will_make_large_single_gift_next_365d",
        "label_will_open_next_campaign_email",
        "label_will_click_next_campaign_email",
        "label_will_unsubscribe_next_90d",
        "label_will_visit_site_next_30d",
        "label_will_view_local_projects_next_90d",
        "label_will_login_next_30d",
        "label_will_give_local_next_365d",
        "label_will_give_to_teacher_previously_supported_next_365d",
        "label_will_give_to_new_schools_next_365d",
        "label_will_give_in_back_to_school_season",
        "label_will_give_during_eoy",
        "label_will_be_event_responsive_next_12m",
        "label_will_use_account_credit_next_90d",
        "label_will_make_gift_card_purchase_next_12m",
    ]

    # Count labels
    count_labels = [
        "label_num_gifts_next_365d",
        "label_num_green_gifts_next_365d",
        "label_num_sessions_next_90d",
        "label_project_page_views_next_90d",
    ]

    # Dollar amount labels
    amount_labels = [
        "label_daf_vehicle_revenue_next_365d",
        "label_green_revenue_next_365d",
        "label_project_revenue_next_365d",
        "label_monthly_revenue_next_365d",
        "label_grand_revenue_next_365d",
        "label_max_single_gift_next_365d",
        "label_monthly_amt_next_6m",
        "label_daf_revenue_next_365d",
    ]

    # Rate/fraction labels (0–1)
    rate_labels = [
        "label_email_open_rate_next_90d",
        "label_email_click_rate_next_90d",
        "label_pct_future_dollars_to_top_category_next_365d",
        "label_optional_donation_rate_on_next_gift",
    ]

    # Ratio labels where 1.0 = "no change"
    ratio_labels = [
        "label_revenue_growth_ratio_next_year",
    ]

    # Time-to-event labels
    time_to_event_labels = [
        "label_time_to_next_gift_days",
    ]

    # Categorical labels
    categorical_labels = [
        "label_next_gift_category",
        "label_next_gift_grade_band",
        "label_quarter_of_max_future_giving_next_12m",
    ]

    # Binary labels whose window may not be fully observed (v1.3.3): impute 0 only when
    # the label has observed values; an all-NaN column means the window runs past the data.
    observability_gated_binary_labels = [
        "label_will_make_major_gift_next_24m",
    ]

    # Apply imputations
    for col in observability_gated_binary_labels:
        if col in features.columns and features[col].notna().any():
            features[col] = features[col].fillna(0.0).astype(float)

    for col in binary_labels:
        if col in features.columns:
            features[col] = features[col].fillna(0.0).astype(float)

    for col in count_labels:
        if col in features.columns:
            features[col] = features[col].fillna(0.0).astype(float)

    for col in amount_labels:
        if col in features.columns:
            features[col] = features[col].fillna(0.0).astype(float)

    for col in rate_labels:
        if col in features.columns:
            features[col] = features[col].fillna(0.0).astype(float)

    for col in ratio_labels:
        if col in features.columns:
            features[col] = features[col].fillna(1.0).astype(float)

    for col in time_to_event_labels:
        if col in features.columns:
            features[col] = features[col].fillna(TIME_TO_NEXT_GIFT_CENSOR_DAYS).astype(float)

    for col in categorical_labels:
        if col in features.columns:
            features[col] = features[col].astype(object).fillna("NONE").astype(str)

    return features

### 4.3 Build Features & Merge Labels

In [29]:
def build_features(
    df_dpr,
    df_email,
    df_site,
    df_monthly,
    df_share,
    df_zip_acs,
    df_project_dates,
    E,
    T,
    H,
    df_credit=None,
    apply_loyalist_rule_at_T=False,
    max_memory_pct=85
):
    """
    Build complete donor-level feature matrix as of time T.
    
    This is the main entry point for feature engineering. It:
    1. Normalizes all timestamps
    2. Filters data to events before T
    3. Creates windowed subsets (3m, 12m, 36m)
    4. Calls helper functions for each feature group
    5. Optionally builds labels for horizon H
    
    IMPORTANT: This does NOT do final imputation or encoding.
    You should call finalize_features() after this to handle:
    - Missing value imputation
    - Categorical encoding
    - Feature scaling (if desired)
    """

    # =====================================================================
    # SETUP: Convert dates and normalize T
    # =====================================================================

    check_system_memory("build_features start", max_used_pct=max_memory_pct)

    T = pd.to_datetime(T)

    # Convert date columns only if needed.
    # Avoid full dataframe copies here unless absolutely necessary.

    if not pd.api.types.is_datetime64_any_dtype(df_dpr["payment_date"]):
        df_dpr["payment_date"] = pd.to_datetime(df_dpr["payment_date"])

    if not pd.api.types.is_datetime64_any_dtype(df_monthly["monthly_subscription_joined_date"]):
        df_monthly["monthly_subscription_joined_date"] = pd.to_datetime(
            df_monthly["monthly_subscription_joined_date"]
        )

    if not pd.api.types.is_datetime64_any_dtype(df_monthly["monthly_subscription_retired_date"]):
        df_monthly["monthly_subscription_retired_date"] = pd.to_datetime(
            df_monthly["monthly_subscription_retired_date"]
        )

    # v1.3: the monthly extract is payment-level; its charge date is
    # monthly_subscription_payment_date. Without this, every charge_date branch
    # (monthly features AND monthly/grand revenue labels) silently did nothing.
    if "charge_date" not in df_monthly.columns and "monthly_subscription_payment_date" in df_monthly.columns:
        df_monthly["charge_date"] = pd.to_datetime(df_monthly["monthly_subscription_payment_date"])

    if "charge_date" in df_monthly.columns:
        if not pd.api.types.is_datetime64_any_dtype(df_monthly["charge_date"]):
            df_monthly["charge_date"] = pd.to_datetime(df_monthly["charge_date"])

    check_system_memory("after setup date conversions", max_used_pct=max_memory_pct)

    # =====================================================================
    # ELIGIBILITY FILTERING (if requested)
    # =====================================================================

    if E is not None:
        eligibility_start = T - pd.DateOffset(months=E)

        # Donors with project donations in eligibility window
        active_project_donors = df_dpr[
            (df_dpr["payment_date"] >= eligibility_start) &
            (df_dpr["payment_date"] < T)
        ]["donor_id"].unique()

        # v1.3: re-apply the SQL loyalist rule as of T (the extract's cohort CTE uses
        # GETDATE(), so it admits donors who qualify only through post-T giving).
        if apply_loyalist_rule_at_T:
            n_before = len(active_project_donors)
            active_project_donors = (
                pd.Index(active_project_donors).intersection(_loyalist_ids_at_T(df_dpr, T)).to_numpy()
            )
            print(f"loyalist rule at T: {n_before:,} -> {len(active_project_donors):,} project donors")

        check_system_memory("after active_project_donors", max_used_pct=max_memory_pct)

        # Donors active in monthly program during eligibility window
        active_monthly_donors = df_monthly[
            (df_monthly["monthly_subscription_joined_date"] < T) &
            (
                df_monthly["monthly_subscription_retired_date"].isna() |
                (df_monthly["monthly_subscription_retired_date"] >= eligibility_start)
            )
        ]["donor_id"].unique()

        check_system_memory("after active_monthly_donors", max_used_pct=max_memory_pct)

        # Union of active donors
        base_donor_ids = pd.Index(
            pd.unique(
                pd.concat(
                    [
                        pd.Series(active_project_donors),
                        pd.Series(active_monthly_donors),
                    ],
                    ignore_index=True,
                )
            ),
            name="donor_id",
        )

        del active_project_donors, active_monthly_donors
        gc.collect()

    else:
        # Original behavior: all donors from both sources
        base_donor_ids = pd.Index(
            pd.unique(
                pd.concat(
                    [
                        df_dpr["donor_id"],
                        df_monthly["donor_id"],
                    ],
                    ignore_index=True,
                )
            ),
            name="donor_id",
        )

    check_system_memory("after base_donor_ids", max_used_pct=max_memory_pct)
    print(f"base_donor_ids: {len(base_donor_ids):,} donors")

    # Create master feature dataframe
    features = pd.DataFrame(index=base_donor_ids)
    feature_groups = {}   # v1.3.5: feature -> group, for the frame manifest

    check_system_memory("after empty features shell", max_used_pct=max_memory_pct)
    print_memory(features, "features empty shell")

    # Restrict all event tables to the base donor universe.
    # These are intentional copies because the downstream function mutates/filters them.

    df_dpr = df_dpr[df_dpr["donor_id"].isin(base_donor_ids)].copy()
    check_system_memory("after df_dpr donor filter", max_used_pct=max_memory_pct)
    print_memory(df_dpr, "df_dpr filtered")

    df_email = df_email[df_email["donor_id"].isin(base_donor_ids)].copy()
    check_system_memory("after df_email donor filter", max_used_pct=max_memory_pct)
    print_memory(df_email, "df_email filtered")

    df_site = df_site[df_site["donor_id"].isin(base_donor_ids)].copy()
    check_system_memory("after df_site donor filter", max_used_pct=max_memory_pct)
    print_memory(df_site, "df_site filtered")

    df_monthly = df_monthly[df_monthly["donor_id"].isin(base_donor_ids)].copy()
    check_system_memory("after df_monthly donor filter", max_used_pct=max_memory_pct)
    print_memory(df_monthly, "df_monthly filtered")

    df_share = df_share[df_share["donor_id"].isin(base_donor_ids)].copy()
    check_system_memory("after df_share donor filter", max_used_pct=max_memory_pct)
    print_memory(df_share, "df_share filtered")

    # =====================================================================
    # NORMALIZE DATES & FILTER PRE-T
    # =====================================================================

    # DPR: main donation records
    if not pd.api.types.is_datetime64_any_dtype(df_dpr["payment_date"]):
        df_dpr["payment_date"] = pd.to_datetime(df_dpr["payment_date"])

    dpr_pre_T = df_dpr[df_dpr["payment_date"] < T].copy()

    check_system_memory("after dpr_pre_T", max_used_pct=max_memory_pct)
    print_memory(dpr_pre_T, "dpr_pre_T")

    # Precompute distances once using direct lat/lon -> miles
    if {"donor_lat_long", "school_lat_long"}.issubset(dpr_pre_T.columns):

        # Parse lat/long strings into numeric columns
        dpr_pre_T["donor_lat_long"] = dpr_pre_T["donor_lat_long"].astype(str)
        dpr_pre_T["school_lat_long"] = dpr_pre_T["school_lat_long"].astype(str)

        donor_lat_lon = dpr_pre_T["donor_lat_long"].str.split(",", expand=True)
        school_lat_lon = dpr_pre_T["school_lat_long"].str.split(",", expand=True)

        dpr_pre_T["donor_lat"] = donor_lat_lon[0].astype(float)
        dpr_pre_T["donor_lon"] = donor_lat_lon[1].astype(float)
        dpr_pre_T["school_lat"] = school_lat_lon[0].astype(float)
        dpr_pre_T["school_lon"] = school_lat_lon[1].astype(float)

        del donor_lat_lon, school_lat_lon
        gc.collect()

        # Direct haversine distance from donor to project
        dpr_pre_T["distance_mi"] = haversine_miles(
            dpr_pre_T["donor_lat"],
            dpr_pre_T["donor_lon"],
            dpr_pre_T["school_lat"],
            dpr_pre_T["school_lon"],
        )
    else:
        dpr_pre_T["distance_mi"] = np.nan

    check_system_memory("after distance_mi", max_used_pct=max_memory_pct)
    print_memory(dpr_pre_T, "dpr_pre_T after distance_mi")

    # Site events
    if not pd.api.types.is_datetime64_any_dtype(df_site["activity_date"]):
        df_site["activity_date"] = pd.to_datetime(df_site["activity_date"])

    site_pre_T = df_site[df_site["activity_date"] < T].copy()

    check_system_memory("after site_pre_T", max_used_pct=max_memory_pct)
    print_memory(site_pre_T, "site_pre_T")

    # Share events (monthly aggregates)
    if "share_month_start" not in df_share.columns:
        df_share["share_month_start"] = pd.to_datetime(
            df_share["share_sent_month"]
        ).dt.to_period("M").dt.to_timestamp()
    elif not pd.api.types.is_datetime64_any_dtype(df_share["share_month_start"]):
        df_share["share_month_start"] = pd.to_datetime(
            df_share["share_month_start"]
        ).dt.to_period("M").dt.to_timestamp()

    share_pre_T = df_share[df_share["share_month_start"] < T].copy()

    check_system_memory("after share_pre_T", max_used_pct=max_memory_pct)
    print_memory(share_pre_T, "share_pre_T")

    # Email events (monthly aggregates)
    if "email_month_start" not in df_email.columns:
        df_email["email_month_start"] = pd.to_datetime(
            df_email["email_sent_month"]
        ).dt.to_period("M").dt.to_timestamp()
    elif not pd.api.types.is_datetime64_any_dtype(df_email["email_month_start"]):
        df_email["email_month_start"] = pd.to_datetime(
            df_email["email_month_start"]
        ).dt.to_period("M").dt.to_timestamp()

    email_pre_T = df_email[df_email["email_month_start"] < T].copy()

    check_system_memory("after email_pre_T", max_used_pct=max_memory_pct)
    print_memory(email_pre_T, "email_pre_T")

    # Monthly subscriptions
    md_pre_T = df_monthly[
        df_monthly["monthly_subscription_joined_date"] < T
    ].copy()

    check_system_memory("after md_pre_T", max_used_pct=max_memory_pct)
    print_memory(md_pre_T, "md_pre_T")

    # v1.3: account credit ledger (AccountCredits.csv) — sole source of credit features
    if df_credit is not None and not df_credit.empty:
        credit = prepare_credit_ledger(df_credit)
        credit = credit[credit["donor_id"].isin(base_donor_ids)]
        credit_pre_T = credit[credit["txn_date"] < T]
        print_memory(credit_pre_T, "credit_pre_T")
    else:
        print("WARNING: no df_credit supplied; account-credit features and the credit-use label will be missing")
        credit, credit_pre_T = None, None

    # =====================================================================
    # WINDOW BOUNDARIES
    # =====================================================================
    # These define our lookback periods for windowed features

    W_short_start = T - pd.DateOffset(months=3)   # [T-3m, T)
    W_mid_start = T - pd.DateOffset(months=12)    # [T-12m, T)
    W_long_start = T - pd.DateOffset(months=36)   # [T-36m, T)

    # Create windowed DPR subsets for counts, amounts, velocities.
    # These are views where possible; no .copy() here.

    dpr_3m = dpr_pre_T[dpr_pre_T["payment_date"] >= W_short_start]
    dpr_12m = dpr_pre_T[dpr_pre_T["payment_date"] >= W_mid_start]
    dpr_36m = dpr_pre_T[dpr_pre_T["payment_date"] >= W_long_start]

    check_system_memory("after dpr windows", max_used_pct=max_memory_pct)
    print_memory(dpr_3m, "dpr_3m")
    print_memory(dpr_12m, "dpr_12m")
    print_memory(dpr_36m, "dpr_36m")

    # Intermediate periods for velocity calculations
    dpr_3to12m = dpr_pre_T[
        (dpr_pre_T["payment_date"] >= W_mid_start) &
        (dpr_pre_T["payment_date"] < W_short_start)
    ]

    dpr_12to36m = dpr_pre_T[
        (dpr_pre_T["payment_date"] >= W_long_start) &
        (dpr_pre_T["payment_date"] < W_mid_start)
    ]

    check_system_memory("after dpr velocity windows", max_used_pct=max_memory_pct)

    # Email and site windows
    email_12m = email_pre_T[email_pre_T["email_month_start"] >= W_mid_start]
    email_3m = email_pre_T[email_pre_T["email_month_start"] >= W_short_start]
    site_3m = site_pre_T[site_pre_T["activity_date"] >= W_short_start]
    share_12m = share_pre_T[share_pre_T["share_month_start"] >= W_mid_start]

    check_system_memory("after email/site/share windows", max_used_pct=max_memory_pct)
    print_memory(email_12m, "email_12m")
    print_memory(email_3m, "email_3m")
    print_memory(site_3m, "site_3m")
    print_memory(share_12m, "share_12m")

    # =====================================================================
    # BUILD FEATURE GROUPS
    # =====================================================================
    # Each join adds a group of related features.
    # Using left joins to preserve all donor_ids.
    #
    # Pattern:
    #   1. build one feature block
    #   2. check/print its memory
    #   3. join to master feature matrix
    #   4. delete the temporary block
    # =====================================================================

    # 1. Identity & ZIP / ACS demographics
    f_identity = _identity_and_zip_features(dpr_pre_T, df_zip_acs, md_pre_T)
    check_system_memory("after _identity_and_zip_features", max_used_pct=max_memory_pct)
    print_memory(f_identity, "f_identity")

    features = _join_group(features, f_identity, "identity", feature_groups)
    del f_identity
    gc.collect()

    check_system_memory("after joining identity features", max_used_pct=max_memory_pct)
    print_memory(features, "features after identity")

    # 2. Lifetime giving behavior & tenure
    f_lifetime = _lifetime_giving_features(dpr_pre_T, T)
    check_system_memory("after _lifetime_giving_features", max_used_pct=max_memory_pct)
    print_memory(f_lifetime, "f_lifetime")

    features = _join_group(features, f_lifetime, "lifetime", feature_groups)
    del f_lifetime
    gc.collect()

    check_system_memory("after joining lifetime features", max_used_pct=max_memory_pct)
    print_memory(features, "features after lifetime")

    # 3. Windowed giving & velocity trends
    f_windowed = _windowed_giving_features(
        dpr_pre_T,
        dpr_3m,
        dpr_12m,
        dpr_36m,
        dpr_3to12m,
        dpr_12to36m,
        T,
    )
    check_system_memory("after _windowed_giving_features", max_used_pct=max_memory_pct)
    print_memory(f_windowed, "f_windowed")

    features = _join_group(features, f_windowed, "windowed", feature_groups)
    del f_windowed
    gc.collect()

    check_system_memory("after joining windowed features", max_used_pct=max_memory_pct)
    print_memory(features, "features after windowed")

    # 4. Channel/payment type mix
    f_channel = _channel_mix_features(dpr_pre_T, dpr_12m)
    check_system_memory("after _channel_mix_features", max_used_pct=max_memory_pct)
    print_memory(f_channel, "f_channel")

    features = _join_group(features, f_channel, "channel", feature_groups)
    del f_channel
    gc.collect()

    check_system_memory("after joining channel features", max_used_pct=max_memory_pct)
    print_memory(features, "features after channel")

    # 5. Monthly subscription program
    f_monthly = _monthly_features(md_pre_T, dpr_pre_T, dpr_12m, T)
    check_system_memory("after _monthly_features", max_used_pct=max_memory_pct)
    print_memory(f_monthly, "f_monthly")

    features = _join_group(features, f_monthly, "monthly", feature_groups)
    del f_monthly
    gc.collect()

    check_system_memory("after joining monthly features", max_used_pct=max_memory_pct)
    print_memory(features, "features after monthly")

    # 6. Teacher/school/content preferences
    f_teacher_school = _teacher_school_features(dpr_pre_T)
    check_system_memory("after _teacher_school_features", max_used_pct=max_memory_pct)
    print_memory(f_teacher_school, "f_teacher_school")

    features = _join_group(features, f_teacher_school, "teacher_school", feature_groups)
    del f_teacher_school
    gc.collect()

    check_system_memory("after joining teacher/school features", max_used_pct=max_memory_pct)
    print_memory(features, "features after teacher/school")

    # 7. Seasonality & rhythm patterns
    f_seasonality = _seasonality_features(dpr_pre_T)
    check_system_memory("after _seasonality_features", max_used_pct=max_memory_pct)
    print_memory(f_seasonality, "f_seasonality")

    features = _join_group(features, f_seasonality, "seasonality", feature_groups)
    del f_seasonality
    gc.collect()

    check_system_memory("after joining seasonality features", max_used_pct=max_memory_pct)
    print_memory(features, "features after seasonality")

    # 8. Email engagement
    f_email = _email_features(email_pre_T, email_3m, email_12m, T)
    check_system_memory("after _email_features", max_used_pct=max_memory_pct)
    print_memory(f_email, "f_email")

    features = _join_group(features, f_email, "email", feature_groups)
    del f_email
    gc.collect()

    check_system_memory("after joining email features", max_used_pct=max_memory_pct)
    print_memory(features, "features after email")

    # 9. Site behavior
    f_site = _site_features(site_pre_T, site_3m, T)
    check_system_memory("after _site_features", max_used_pct=max_memory_pct)
    print_memory(f_site, "f_site")

    features = _join_group(features, f_site, "site", feature_groups)
    del f_site
    gc.collect()

    check_system_memory("after joining site features", max_used_pct=max_memory_pct)
    print_memory(features, "features after site")

    # 10. Share events
    f_share = _share_features(share_pre_T, share_12m, T)
    check_system_memory("after _share_features", max_used_pct=max_memory_pct)
    print_memory(f_share, "f_share")

    features = _join_group(features, f_share, "share", feature_groups)
    del f_share
    gc.collect()

    check_system_memory("after joining share features", max_used_pct=max_memory_pct)
    print_memory(features, "features after share")

    # 11. Project outcomes & matching
    f_project_outcome = _project_outcome_features(dpr_pre_T)
    check_system_memory("after _project_outcome_features", max_used_pct=max_memory_pct)
    print_memory(f_project_outcome, "f_project_outcome")

    features = _join_group(features, f_project_outcome, "project_outcome", feature_groups)
    del f_project_outcome
    gc.collect()

    check_system_memory("after joining project outcome features", max_used_pct=max_memory_pct)
    print_memory(features, "features after project outcome")

    # 12. School/teacher availability
    if df_project_dates is not None and not df_project_dates.empty:
        horizon = H if H is not None else pd.Timedelta(days=365)

        f_future_opp = _future_opportunity_features(
            dpr_pre_T,
            df_project_dates,
            T,
            horizon,
        )
        check_system_memory("after _future_opportunity_features", max_used_pct=max_memory_pct)
        print_memory(f_future_opp, "f_future_opp")

        features = _join_group(features, f_future_opp, "future_opp", feature_groups)
        for _c in f_future_opp.columns:
            features[_c] = features[_c].fillna(9999 if _c.endswith('days_since_posted') else 0)
        del f_future_opp
        gc.collect()
    else:
        for _c in ['top_school_open_project_at_T', 'top_teacher_open_project_at_T',
                   'top_school_projects_posted_12m', 'top_teacher_projects_posted_12m']:
            features[_c] = 0
        features['top_school_days_since_posted'] = 9999
        features['top_teacher_days_since_posted'] = 9999

    check_system_memory("after joining future opportunity features", max_used_pct=max_memory_pct)
    print_memory(features, "features after future opportunity")

    # 13. Latest donation
    f_latest = _latest_donation_features(
        dpr_pre_T=dpr_pre_T,
        df_share=df_share,
        T=T,
    )
    check_system_memory("after _latest_donation_features", max_used_pct=max_memory_pct)
    print_memory(f_latest, "f_latest")

    features = _join_group(features, f_latest, "latest", feature_groups)
    del f_latest
    gc.collect()

    check_system_memory("after joining latest donation features", max_used_pct=max_memory_pct)
    print_memory(features, "features after latest donation")

    # 14. v1.3 behavioral expansion
    f_expansion = _expansion_features(dpr_pre_T, T)
    print_memory(f_expansion, "f_expansion")
    features = _join_group(features, f_expansion, "expansion", feature_groups)
    del f_expansion
    gc.collect()

    f_email_type = _email_type_features(email_12m)
    features = _join_group(features, f_email_type, "email_type", feature_groups)
    del f_email_type

    f_lifecycle = _lifecycle_features(dpr_pre_T, md_pre_T, T)
    features = _join_group(features, f_lifecycle, "lifecycle", feature_groups)
    del f_lifecycle
    gc.collect()

    check_system_memory("after v1.3 expansion features", max_used_pct=max_memory_pct)
    print_memory(features, "features after v1.3 expansion")

    # 15. v1.3 account credit ledger
    if credit is not None:
        f_credit = _credit_features(credit_pre_T, dpr_pre_T, T, features.index)
        features = _join_group(features, f_credit, "credit", feature_groups)
        del f_credit
        print_memory(features, "features after credit ledger")

    # v1.3.3: drop current-state (leaky) features from the output
    features = features.drop(columns=[c for c in SNAPSHOT_DROPPED_FEATURES if c in features.columns])

    # =====================================================================
    # BUILD LABELS
    # =====================================================================

    labels = _build_labels(
        df_dpr=df_dpr,
        df_monthly=df_monthly,
        df_share=df_share,
        df_email=df_email,
        df_site=df_site,
        T=T,
        donor_index=features.index,
        df_credit=credit,
    )
    check_system_memory("after _build_labels", max_used_pct=max_memory_pct)
    print_memory(labels, "labels")

    label_groups = dict(labels.attrs.get("label_groups", {}))
    features = features.join(labels, how="left")
    del labels
    gc.collect()

    check_system_memory("after joining labels", max_used_pct=max_memory_pct)
    print_memory(features, "features after labels")

    features = _apply_label_imputation(features)

    check_system_memory("after _apply_label_imputation", max_used_pct=max_memory_pct)
    print_memory(features, "features after label imputation")

    # Drop raw ZIP column if present
    if "donor_zip5" in features.columns:
        features = features.drop(columns=["donor_zip5"])

    check_system_memory("after dropping donor_zip5", max_used_pct=max_memory_pct)

    # Add donor_id as an explicit column.
    # Avoid full features.copy() here because this can duplicate the final feature matrix.
    features.insert(0, "donor_id", features.index)
    features = features.reset_index(drop=True)

    check_system_memory("build_features complete", max_used_pct=max_memory_pct)
    print_memory(features, "features final")

    # v1.3.5: provenance for the frame manifest (see 4.4)
    features.attrs["frame_meta"] = {
        "T": T,
        "feature_groups": {c: g for c, g in feature_groups.items() if c in features.columns},
        "label_groups": {c: g for c, g in label_groups.items() if c in features.columns},
    }
    return features

### 4.4 Frame Manifests (v1.3.5)

Sidecar metadata the modeling notebook requires: source history, feature lookbacks and flags, label windows and maturity.

In [30]:
# =============================================================================
# 4.4 FRAME MANIFESTS (v1.3.5)
#
# Every frame CSV gets a <stem>.manifest.json sidecar that the modeling notebook
# requires. It makes feature eligibility and label maturity deterministic:
#
#   frame     : name, as-of date T, extract DATA_ASOF, row count
#   sources   : source -> history_start / data_end (from the loaded extracts)
#   features  : feature -> group, sources, lookback_days, available_at_scoring,
#               sensitive, snapshot, model_input
#   labels    : label -> window_days, sources, observed_through, mature
#
# A feature is usable at T only if T - lookback_days >= history_start for each of
# its sources. A label is mature at T only if T + window_days <= observed_through + 1d.
# Missing information is never treated as observed (v1.3.6): a dynamic source with no
# data has history_start / data_end = null, and any label reading it is NOT mature.
# Static sources (no time dimension) are flagged "static": true.
# A <stem>.features.csv copy of the feature table is written for review.
# =============================================================================

import json
import math
import re
from datetime import datetime

FEATURES_NOTEBOOK_VERSION = "1.3.7"

# Source extracts: date column and grain (monthly grains end at the month's last day)
SOURCE_DATE_COLUMNS = {
    "dpr":           ("payment_date", "daily"),
    "email":         ("email_sent_month", "monthly"),
    "site":          ("activity_date", "daily"),
    "monthly":       ("monthly_subscription_payment_date", "daily"),
    "share":         ("share_sent_month", "monthly"),
    "project_dates": ("project_last_posted_date", "daily"),
    "credit":        ("txn_date", "daily"),
}
STATIC_SOURCES = ["zip_acs"]          # no time dimension

# Feature group (as joined in build_features) -> sources it reads
FEATURE_GROUP_SOURCES = {
    "identity":        ["dpr", "monthly", "zip_acs"],
    "lifetime":        ["dpr"],
    "windowed":        ["dpr"],
    "channel":         ["dpr"],
    "monthly":         ["monthly", "dpr"],
    "teacher_school":  ["dpr"],
    "seasonality":     ["dpr"],
    "email":           ["email"],
    "site":            ["site"],
    "share":           ["share"],
    "project_outcome": ["dpr"],
    "future_opp":      ["dpr", "project_dates"],
    "latest":          ["dpr", "share"],
    "expansion":       ["dpr"],
    "email_type":      ["email"],
    "lifecycle":       ["dpr", "monthly"],
    "credit":          ["credit"],
}

# Label group (as joined in _build_labels) -> sources it reads
LABEL_GROUP_SOURCES = {
    "core":            ["dpr"],
    "revenue":         ["dpr", "monthly"],
    "monthly_program": ["monthly"],
    "mid_major":       ["dpr"],
    "email":           ["email"],
    "site":            ["site", "dpr"],
    "preference":      ["dpr"],
    "seasonality":     ["dpr"],
    "operational":     ["dpr"],
}
LABEL_SOURCE_OVERRIDES = {
    "label_will_be_midlevel_grand_next_12m": ["dpr", "monthly"],   # grand = project + monthly
    "label_will_use_account_credit_next_90d": ["credit"],
}

# Label windows: parsed from "_next_<n>d", "_next_<n>m", "_next_year"; explicit otherwise
LABEL_MONTH_WINDOW_DAYS = {6: 180, 12: 365, 24: 730}     # matches _future_window calls
LABEL_WINDOW_OVERRIDES = {
    "label_time_to_next_gift_days": 365,
    "label_next_gift_category": 365,
    "label_next_gift_grade_band": 365,
    "label_optional_donation_rate_on_next_gift": 365,
    "label_will_open_next_campaign_email": 90,
    "label_will_click_next_campaign_email": 90,
    "label_will_give_in_back_to_school_season": 365,
    "label_will_give_during_eoy": 365,
}

# Feature lookbacks: parsed from "_<n>m", "<a>to<b>" (months) and "last2yr"; explicit otherwise
LOOKBACK_OVERRIDES = {
    "is_continuing_donor_12m": 730,   # needs giving in both 0-12m and 12-24m
    "is_reactivated_12m": 730,        # needs history before 24m
}
NON_MODEL_INPUT_COLUMNS = ["first_donation_date", "last_donation_date"]   # raw dates
SENSITIVE_FEATURES = []   # frame features flagged sensitive (demographics are flagged in modeling)


def _lookback_days(feature):
    if feature in LOOKBACK_OVERRIDES:
        return LOOKBACK_OVERRIDES[feature]
    months = [int(n) for n in re.findall(r"_(\d+)m(?=_|$)", feature)]
    months += [int(b) for _, b in re.findall(r"(\d+)to(\d+)", feature)]
    if "last2yr" in feature:
        months.append(24)
    return int(math.ceil(max(months) * 365 / 12)) if months else None


def _label_window_days(label):
    if label in LABEL_WINDOW_OVERRIDES:
        return LABEL_WINDOW_OVERRIDES[label]
    m = re.search(r"_next_(\d+)d$", label)
    if m:
        return int(m.group(1))
    m = re.search(r"_next_(\d+)m$", label)
    if m and int(m.group(1)) in LABEL_MONTH_WINDOW_DAYS:
        return LABEL_MONTH_WINDOW_DAYS[int(m.group(1))]
    if label.endswith("_next_year"):
        return 365
    raise ValueError(f"No label window defined for '{label}'; add it to LABEL_WINDOW_OVERRIDES")


def build_source_meta(data_asof, **extracts):
    """history_start / data_end per source, from the raw extracts (not modified).
    data_end is capped at the extract's DATA_ASOF: nothing later counts as observed."""
    data_asof = pd.Timestamp(data_asof)
    meta = {}
    for name, (col, grain) in SOURCE_DATE_COLUMNS.items():
        unknown = {"history_start": None, "data_end": None, "static": False}
        df = extracts.get(name)
        if df is None or df.empty or (name != "credit" and col not in df.columns):
            meta[name] = unknown
            continue
        if name == "credit":
            dates = prepare_credit_ledger(df)["txn_date"]
        else:
            dates = pd.to_datetime(df[col], errors="coerce")
        dates = dates.dropna()
        if dates.empty:
            meta[name] = unknown
            continue
        start, end = dates.min(), dates.max()
        if grain == "monthly":
            end = end + pd.offsets.MonthEnd(0)
        end = min(end, data_asof)
        meta[name] = {"history_start": start.date().isoformat(), "data_end": end.date().isoformat(), "static": False}
    for name in STATIC_SOURCES:
        meta[name] = {"history_start": None, "data_end": None, "static": True}
    return meta


def frame_manifest(features, name, T, data_asof, source_meta, file_name):
    meta = features.attrs.get("frame_meta")
    if not meta:
        raise ValueError("features has no frame_meta; build it with build_features() v1.3.5+")
    feature_cols = [c for c in features.columns if c != "donor_id" and not c.startswith("label_")]
    label_cols = [c for c in features.columns if c.startswith("label_")]

    missing = [c for c in feature_cols if c not in meta["feature_groups"]]
    if missing:
        raise ValueError(f"Features without a recorded group: {missing[:10]}")

    feature_rows = []
    for c in feature_cols:
        group = meta["feature_groups"][c]
        feature_rows.append({
            "feature": c,
            "group": group,
            "sources": FEATURE_GROUP_SOURCES[group],
            "lookback_days": _lookback_days(c),
            "available_at_scoring": True,          # every feature is computed as of T
            "sensitive": c in SENSITIVE_FEATURES,
            "snapshot": c in SNAPSHOT_FEATURES,
            "model_input": c not in NON_MODEL_INPUT_COLUMNS,
        })

    labels = {}
    for c in label_cols:
        group = meta["label_groups"].get(c)
        if group is None:
            raise ValueError(f"Label without a recorded group: {c}")
        sources = LABEL_SOURCE_OVERRIDES.get(c, LABEL_GROUP_SOURCES[group])
        window = _label_window_days(c)
        # Every dynamic source must have a data_end; one missing source makes the label
        # unobserved (never "mature" by ignoring it).
        dynamic = [s for s in sources if not source_meta[s]["static"]]
        missing = [s for s in dynamic if not source_meta[s]["data_end"]]
        if missing or not dynamic:
            observed_through, mature = None, False
        else:
            observed_through = min(pd.Timestamp(source_meta[s]["data_end"]) for s in dynamic)
            mature = bool(T + pd.Timedelta(days=window) <= observed_through + pd.Timedelta(days=1))
        labels[c] = {
            "group": group,
            "window_days": window,
            "sources": sources,
            "missing_sources": missing,
            "observed_through": observed_through.date().isoformat() if observed_through is not None else None,
            "mature": mature,
        }

    return {
        "manifest_version": 1,
        "features_notebook_version": FEATURES_NOTEBOOK_VERSION,
        "frame": {
            "name": name,
            "file": file_name,
            "data_asof": pd.Timestamp(T).date().isoformat(),
            "extract_data_asof": pd.Timestamp(data_asof).date().isoformat(),
            "n_rows": int(len(features)),
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
        "sources": source_meta,
        "features": feature_rows,
        "labels": labels,
    }


def write_frame(features, csv_path, name, T, data_asof, source_meta):
    """Write the frame CSV plus <stem>.manifest.json and <stem>.features.csv beside it.
    The caller chooses the file name (naming convention lives in the run cell)."""
    csv_path = Path(csv_path)
    manifest = frame_manifest(features, name, T, data_asof, source_meta, csv_path.name)
    features.to_csv(csv_path, index=False)
    with open(csv_path.with_name(f"{csv_path.stem}.manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    pd.DataFrame(manifest["features"]).to_csv(csv_path.with_name(f"{csv_path.stem}.features.csv"), index=False)
    n_mature = sum(v["mature"] for v in manifest["labels"].values())
    print(f"wrote {csv_path.name}: {len(features):,} rows, {len(manifest['features'])} features, "
          f"{n_mature}/{len(manifest['labels'])} labels mature at T={pd.Timestamp(T).date()}")
    return csv_path


## 5. Feature & Label Generation

In [31]:
def print_memory(df, name):
    memory_gb = df.memory_usage(deep=True).sum() / 1024**3
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} cols | {memory_gb:.2f} GB")


def check_system_memory(step_name, max_used_pct=85):
    """
    Stop gracefully if system RAM usage gets too high.

    This will not interrupt a single giant pandas operation mid-line, but it
    will stop the notebook between major feature-building steps before memory
    pressure gets too dangerous.
    """
    vm = psutil.virtual_memory()

    used_pct = vm.percent
    used_gb = vm.used / 1024**3
    total_gb = vm.total / 1024**3
    available_gb = vm.available / 1024**3

    print(
        f"[memory check] {step_name}: "
        f"{used_pct:.1f}% used | "
        f"{used_gb:.2f} GB used / {total_gb:.2f} GB total | "
        f"{available_gb:.2f} GB available"
    )

    if used_pct >= max_used_pct:
        raise MemoryError(
            f"Stopping gracefully at '{step_name}' because system memory is "
            f"{used_pct:.1f}% used, above threshold {max_used_pct}%."
        )

In [32]:
# 1. Load data
loadPATH  = Path('/Users/matt.fritz/Desktop/Donor Predictive Models/SQL Donor Data')
writePATH = Path('/Users/matt.fritz/Desktop/Donor Predictive Models/Constructed Data')

df_dpr           = pd.read_csv(loadPATH / 'DonorProjectRecords_last4yr.csv')
df_email         = pd.read_csv(loadPATH / 'Email Events 36mo.csv')
df_site          = pd.read_csv(loadPATH / 'Site Events FY24-pres.csv')
df_monthly       = pd.read_csv(loadPATH / 'Monthly Donation Subscr All Time.csv')
df_share         = pd.read_csv(loadPATH / 'Share Events All Time.csv')
df_project_dates = pd.read_csv(loadPATH / 'Project Dates FY22-pres.csv')
df_zip_acs       = pd.read_csv(loadPATH / 'Merged_Zip_ACS_Demographics.csv')
df_credit        = pd.read_csv(loadPATH / 'AccountCredits.csv')

# v1.3 options
APPLY_LOYALIST_RULE_AT_T = True   # re-apply the cohort rule as of each T (required with the v1.3 local SQL)
RUN_INPUT_CHECKS         = True

if RUN_INPUT_CHECKS:
    verify_expansion_inputs(df_dpr)
    verify_credit_ledger(df_credit, df_dpr)

# 2. Define reference time, horizon, and eligibility window
DATA_ASOF = pd.to_datetime(df_dpr['payment_date']).max().normalize()

# v1.3.5: history start / data end per source, recorded in every frame manifest
SOURCE_META = build_source_meta(
    DATA_ASOF,
    dpr=df_dpr, email=df_email, site=df_site, monthly=df_monthly,
    share=df_share, project_dates=df_project_dates, credit=df_credit,
)

E = 12                                   # Only donors who gave in E months prior to T are eligible
T = DATA_ASOF - pd.Timedelta(days=365)   # Training features end date (labels close at DATA_ASOF)
H = pd.Timedelta(days=365)               # Label prediction horizon
O = pd.Timedelta(days=365)               # Out-of-time lag; OOT features land on DATA_ASOF

_d = lambda d: f"{d:%Y%b%d}".lower()
FRAME_TAG = f"t{_d(T)}_oot{_d(T + O)}-{_d(T + O + H)}"
print(f'DATA_ASOF={DATA_ASOF.date()}  train T={T.date()}  OOT T={(T + O).date()}  tag={FRAME_TAG}')
'''
                                    ELIGIBILITY
                              T-E | ----------- | T
                                  
                              TRAINING FEATURES       TRAINING LABEL
  | ALL HISTORY ------------------------------- | T | -------------- | T+H
                                                      
                                                      OOT LAG
                                                  T | -- O -- |
                                                    
                                                 OOT FEATURES           OOT LABEL
  | ALL HISTORY --------------------------------------------- | T+O | -------------- | T+O+H
'''
# 3. Build and write each frame: CSV + <stem>.manifest.json + <stem>.features.csv
#    train : T          oot : T + O
#    score : DATA_ASOF  (only when T + O is earlier; otherwise the oot frame IS the current frame)
import gc

BUILD_ARGS = dict(
    df_dpr=df_dpr,
    df_email=df_email,
    df_site=df_site,
    df_monthly=df_monthly,
    df_share=df_share,
    df_project_dates=df_project_dates,
    df_zip_acs=df_zip_acs,
    E=E,
    H=H,
    df_credit=df_credit,
    apply_loyalist_rule_at_T=APPLY_LOYALIST_RULE_AT_T,
)

FRAMES = {
    "train": (T,     writePATH / f"train_features_{FRAME_TAG}.csv"),
    "oot":   (T + O, writePATH / f"oot_features_{FRAME_TAG}.csv"),
}
if T + O < DATA_ASOF:
    FRAMES["score"] = (DATA_ASOF, writePATH / f"score_features_t{_d(DATA_ASOF)}.csv")

for frame_name, (frame_T, frame_path) in FRAMES.items():
    frame = build_features(T=frame_T, **BUILD_ARGS)
    write_frame(frame, frame_path, frame_name, frame_T, DATA_ASOF, SOURCE_META)
    del frame
    gc.collect()

# 4. Which files go where in the modeling notebook (per-label maturity is enforced there)
_oot_labeled = T + O + H <= DATA_ASOF
_current = FRAMES["score"][1] if "score" in FRAMES else FRAMES["oot"][1]
_refit = FRAMES["oot"][1] if _oot_labeled else FRAMES["train"][1]
print("\nModeling notebook frame paths:")
if _oot_labeled:
    print(f"  VALIDATE_FRAMES = {{'train': FRAME_DIR / '{FRAMES['train'][1].name}',\n"
          f"                     'validation': FRAME_DIR / '{FRAMES['oot'][1].name}', 'test': None}}")
else:
    print(f"  (no validation frame: OOT labels close {(T + O + H).date()}, after DATA_ASOF {DATA_ASOF.date()}; "
          f"set T = DATA_ASOF - 730 days to build train + validation + score in one run)")
print(f"  SCORE_FRAMES    = {{'train': FRAME_DIR / '{_refit.name}',\n"
      f"                     'score': FRAME_DIR / '{_current.name}'}}")


payment_id duplicated: 0.00%  (should be 0)
checkouts with >1 distinct referral_source: 0.00%  (occasion-level shares assume ~0)
checkouts with >1 distinct payment_type: 2.44%  (payment type is shared at the gift level, so this is informational)
gifts dated after the project's current expiration: 0.01% of rows with a date
rows dropped for unparseable donor/date: 0.00%
ledger rows=9,020,895 donors=2,818,342 dates 2003-04-01 → 2026-09-30
rows per donor-day: max=1  (1 = daily grain)
net_change == credits - debits: 100.00%
balance continuity (prev + net == eod): 100.00%
negative balances: 0.18%
ledger debit days matching DPR non-green $ on the same donor-day: 27.30%  (n=6,972,453)
DATA_ASOF=2026-09-30  train T=2025-09-30  OOT T=2026-09-30  tag=t2025sep30_oot2026sep30-2027sep30
[memory check] build_features start: 74.7% used | 17.83 GB used / 64.00 GB total | 16.18 GB available
[memory check] after setup date conversions: 74.7% used | 17.83 GB used / 64.00 GB total | 16.18 GB available
loya

## 6. Feature Inventory

> **v1.3 note:** the counts and names below predate v1.3 and were already out of date in v1.2 (the v1.2 run produced 179 features, not 174). See the v1.3 changelog at the top of the notebook for what was added, fixed and flagged. Regenerate this inventory from the output frame after the next run.

## Features (174 total)

### Identity & Demographics (21 features)
Core donor attributes and ZIP-level demographics from ACS data:
- **Identity**: `donor_zip5`, `is_teacher`, `is_teacher_referred`, `is_marketing_subscribed`, `is_major_gift_donor`
- **Account**: `ever_used_account_credit`, `current_account_credit_balance`
- **Donor type flags**: `has_project_history`, `has_monthly_history`, `is_project_only_donor`, `is_monthly_only_donor`, `is_hybrid_donor`
- **ZIP demographics** (9): `zip_log_total_population`, `zip_median_age`, `zip_avg_household_size`, `zip_median_home_value`, `zip_pct_minority`, `zip_pct_households_with_children`, `zip_pct_single_parent`, `zip_pct_in_labor_force`, `zip_unemployment_rate`

---

### Lifetime Giving Behavior (14 features)
Cumulative metrics through time T:
- **Time features**: `first_donation_date`, `last_donation_date`, `tenure_days`, `tenure_years`, `tenure_bucket`
- **Volume**: `lifetime_gift_count`, `lifetime_amount`, `max_donation_sequence_number`
- **Distribution**: `lifetime_median_gift_amount`, `lifetime_max_gift_amount`, `lifetime_cv_gift_amount`
- **Timing patterns**: `mean_gap_between_gifts_days_last2yr`, `cv_gap_between_gifts_days_last2yr`
- **Early behavior**: `pct_early_gifts_in_lifetime`

---

### Windowed Giving & Velocity (26 features)
Time-windowed aggregations (3m, 12m, 36m) and velocity ratios:
- **Counts by window**: `gift_count_3m`, `gift_count_12m`, `gift_count_36m`
- **Green/non-green counts**: `gift_count_green_3m/12m/36m`, `gift_count_nongreen_3m/12m/36m`
- **Amounts by window**: `gift_amount_3m`, `gift_amount_12m`, `gift_amount_36m`
- **Green/non-green amounts**: `gift_amount_green_3m/12m/36m`, `gift_amount_nongreen_3m/12m/36m`
- **Medians**: `median_gift_amount_3m`, `median_gift_amount_12m`, `median_gift_amount_36m`
- **Recency**: `days_since_last_gift`, `days_since_second_to_last_gift`
- **Velocity ratios**: `amount_velocity_0to3_vs_3to12`, `amount_velocity_0to12_vs_12to36`, `count_velocity_0to12_vs_12to36`

---

### Channel & Payment Type Mix (22 features)
Payment channel preferences and special gift types (lifetime and 12m):
- **DAF**: `pct_amount_daf_lifetime/12m`, `pct_count_daf_lifetime/12m`
- **Green payment**: `pct_amount_green_lifetime/12m`, `pct_count_green_lifetime/12m`
- **Gift cards**: `pct_gifts_gift_card_lifetime/12m`, `pct_amount_gift_card_lifetime/12m`
- **Big events**: `pct_amount_big_event_lifetime/12m`, `pct_count_big_event_lifetime/12m`
- **Optional donations**: `avg_optional_donation_rate_lifetime`, `avg_optional_donation_rate_12m`
- **Anonymous**: `pct_gifts_anonymous_lifetime/12m`
- **Classroom essentials**: `pct_amount_classroom_essentials_lifetime/12m`

---

### Monthly Giving Program (10 features)
Monthly subscription behavior:
- **Status**: `is_monthly_donor_current`
- **Volume**: `monthly_lifetime_amount`, `monthly_amount_12m`, `monthly_median_gift_amount`
- **Mix**: `pct_amount_monthly_lifetime`, `pct_amount_monthly_12m`
- **Tenure**: `months_on_program`, `months_since_last_monthly_charge`, `monthly_longest_streak_months`
- **Timing**: `monthly_joined_before_first_project_gift`

---

### Teacher, School & Content Preferences (20 features)
Diversity and concentration in giving targets:
- **Entropy**: `entropy_teacher`, `entropy_school`, `entropy_zip`, `entropy_category`, `entropy_grade`
- **Unique counts**: `num_unique_teachers`, `num_unique_schools`, `num_unique_school_zips`, `num_unique_categories`, `num_unique_grades`
- **Top concentration**: `pct_amount_to_top_teacher`, `pct_amount_to_top_school`, `pct_amount_to_top_category`, `pct_amount_to_top_grade`
- **First/last project loyalty**: `pct_gifts_first_project`, `pct_gifts_last_project`, `pct_amount_first_project`, `pct_amount_last_project`
- **Teacher quality**: `mean_teacher_lifetime_projects_fully_funded`, `mean_teacher_lifetime_donations`

---

### Seasonality Features (10 features)
Temporal patterns in giving:
- **Calendar patterns**: `pct_amount_in_back_to_school`, `pct_amount_in_final_week_of_year`, `pct_amount_on_weekends`
- **Month diversity**: `entropy_gift_month`
- **Peak periods**: `pct_amount_in_top_month`, `pct_amount_in_top_quarter`
- **First gift timing**: `first_donation_month`, `first_donation_quarter`, `first_donation_dow_sin`, `first_donation_dow_cos`

---

### Email Engagement (12 features)
Email behavior and responsiveness (3m and 12m windows):
- **Volume**: `emails_sent_3m/12m`, `emails_opened_3m/12m`, `emails_clicked_3m/12m`
- **Rates**: `email_open_rate_3m/12m`, `email_click_rate_3m/12m`
- **Velocity**: `email_open_rate_velocity_3m_vs_12m`
- **Recency**: `days_since_last_email_sent`

---

### Site Behavior (9 features)
Website activity patterns (3m window):
- **Activity level**: `days_with_any_site_activity_3m`, `avg_sessions_per_active_day_3m`
- **Engagement depth**: `avg_session_duration_min_3m`, `checkout_intent_min_per_session_3m`
- **Recency**: `days_since_last_cart_visit`
- **Content focus**: `campaign_session_share_3m`, `share_project_page_session_pct_3m`, `share_search_page_session_pct_3m`, `share_teacher_page_session_pct_3m`

---

### Share Events (4 features)
Social sharing behavior:
- **Volume**: `share_events_lifetime`, `share_events_12m`
- **Consistency**: `share_active_months_12m`, `share_month_coverage_ratio`

---

### Same School & Teacher Flags (2 features)
Availability indicators for longitudinal targeting:
- `teacher_still_available_during_range`
- `school_still_available_during_range`

---

### Project Outcomes & Matching (8 features)
Project characteristics and outcomes:
- **Geography**: `median_donor_to_project_distance_mi`, `pct_gifts_within_15mi`, `is_local_donor`
- **Project size**: `mean_project_total_cost`, `median_project_total_cost`
- **Outcomes**: `pct_projects_fully_funded`
- **Matching**: `mean_match_multiplier`, `pct_gifts_with_match`

---

### Latest Donation (16 features)
Characteristics of most recent gift as of time T:
- **Geography**: `latest_distance_mi`
- **Amounts**: `latest_gift_amount_green`, `latest_gift_amount_nongreen`
- **Project details**: `latest_project_category`, `latest_project_grade`, `latest_project_total_cost`, `latest_project_got_fully_funded`
- **Gift characteristics**: `latest_payment_type_is_green`, `latest_is_giftcard_purchase`, `latest_optional_donation_percent`, `latest_match_xyi_multiplier`
- **Behavior**: `latest_shared_any`, `latest_referral_channel`
- **Project position**: `latest_donation_is_projects_first`, `latest_donation_is_projects_last`
- **Teacher quality**: `latest_teacher_lifetime_projects_fully_funded`

---

## Labels (50 total, if H provided)

All labels predict future behavior from time T forward. Horizons are fixed per label (not controlled by H parameter).

### Core Giving Propensity (8 labels)
Binary and count predictions for general giving behavior:
- **Binary propensity**: `label_will_give_next_30d`, `label_will_give_next_90d`, `label_will_give_next_365d`
- **Frequency**: `label_num_gifts_next_365d`, `label_will_give_2plus_times_next_365d`
- **Green giving**: `label_num_green_gifts_next_365d`
- **Time-to-event**: `label_time_to_next_gift_days` (censored at 366 days)
- **Special events**: `label_will_make_big_event_gift_next_365d`

---

### Revenue & Value (8 labels)
Dollar amount predictions and revenue growth:
- **Revenue by type**: `label_green_revenue_next_365d`, `label_project_revenue_next_365d`, `label_monthly_revenue_next_365d`, `label_grand_revenue_next_365d`
- **Large gifts**: `label_max_single_gift_next_365d`
- **Growth**: `label_revenue_growth_ratio_next_year` (ratio: future 12m / prior 12m)
- **Thresholds**: `label_will_cross_midlevel_threshold_next_12m` (≥$1,000 grand revenue)
- **Major gifts**: `label_will_make_major_gift_next_24m` (single gift ≥$5,000)

---

### Monthly Program (6 labels)
Monthly subscription lifecycle predictions:
- **Acquisition**: `label_will_start_monthly_next_90d`, `label_will_start_monthly_next_365d`
- **Churn**: `label_will_churn_monthly_next_6m`
- **Changes**: `label_will_upgrade_monthly_next_6m`, `label_will_downgrade_monthly_next_6m`
- **Revenue**: `label_monthly_amt_next_6m`

---

### Midlevel / Major / DAF (5 labels)
High-value donor predictions:
- **DAF**: `label_daf_revenue_next_365d`, `label_will_make_daf_gift_next_365d`, `label_will_shift_to_daf_heavy_next_12m` (≥50% via DAF)
- **Midlevel**: `label_will_be_midlevel_grand_next_12m` (≥$1,000 grand revenue)
- **Large single gifts**: `label_will_make_large_single_gift_next_365d` (≥$500)

---

### Email Engagement (5 labels)
Email response predictions:
- **Aggregate rates**: `label_email_open_rate_next_90d`, `label_email_click_rate_next_90d`
- **Next campaign**: `label_will_open_next_campaign_email`, `label_will_click_next_campaign_email`
- **Unsubscribe**: `label_will_unsubscribe_next_90d`

---

### Site Behavior & Activity (5 labels)
Website engagement predictions:
- **Visit**: `label_will_visit_site_next_30d`
- **Activity level**: `label_num_sessions_next_90d`, `label_project_page_views_next_90d`
- **Special behavior**: `label_will_view_local_projects_next_90d`, `label_will_login_next_30d`

---

### Product & Project Preference (6 labels)
Giving target predictions:
- **Next gift attributes**: `label_next_gift_category`, `label_next_gift_grade_band` (categorical)
- **Geography**: `label_will_give_local_next_365d` (within 25 miles)
- **Concentration**: `label_pct_future_dollars_to_top_category_next_365d`
- **Teacher/school loyalty**: `label_will_give_to_teacher_previously_supported_next_365d`, `label_will_give_to_new_schools_next_365d`

---

### Seasonality & Events (4 labels)
Temporal giving patterns:
- **Seasonal**: `label_will_give_in_back_to_school_season` (Aug-Sep), `label_will_give_during_eoy` (Dec 24-31)
- **Event responsive**: `label_will_be_event_responsive_next_12m` (≥2 gifts on defined events)
- **Peak period**: `label_quarter_of_max_future_giving_next_12m` (Q1/Q2/Q3/Q4, rolling quarters from T)

---

### Operational / Mechanism Behavior (3 labels)
Gift mechanics and special features:
- **Account credit**: `label_will_use_account_credit_next_90d`
- **Optional support**: `label_optional_donation_rate_on_next_gift`
- **Gift cards**: `label_will_make_gift_card_purchase_next_12m`

---

## Summary Statistics

| Category | Count |
|----------|-------|
| **Features** | **174** |
| Identity & Demographics | 21 |
| Lifetime Giving | 14 |
| Windowed Giving | 26 |
| Channel Mix | 22 |
| Monthly Program | 10 |
| Teacher/School Preferences | 20 |
| Seasonality | 10 |
| Email Engagement | 12 |
| Site Behavior | 9 |
| Share Events | 4 |
| Same School/Teacher Flags | 2 |
| Project Outcomes | 8 |
| Latest Donation | 16 |
| **Labels** | **50** |
| Core Giving Propensity | 8 |
| Revenue & Value | 8 |
| Monthly Program | 6 |
| Midlevel / DAF | 5 |
| Email Engagement | 5 |
| Site Behavior | 5 |
| Product Preference | 6 |
| Seasonality & Events | 4 |
| Operational | 3 |
| **TOTAL** | **224** |

---

## Notes

### Time Scopes
- **STATIC**: ZIP demographics, some identity flags
- **LIFETIME**: Cumulative through T (e.g., `lifetime_amount`, `tenure_days`)
- **WINDOWED**: Specific lookbacks (3m, 12m, 36m relative to T)
- **LABELS**: Future periods with fixed horizons (30d, 90d, 365d, 24m from T)

### Label Imputation Rules
Applied by `_apply_label_imputation()`:
- **Binary** → 0.0
- **Count** → 0.0
- **Amount** → 0.0
- **Rate** → 0.0
- **Ratio** → 1.0 (no change baseline)
- **Time-to-event** → 366 (censored)
- **Categorical** → "NONE"

### Label Horizons
Unlike the old 4-label implementation, horizons are now **fixed per label**:
- 30d: Next-month propensity
- 90d: Quarterly targets
- 180d (6m): Monthly program changes
- 365d: Annual revenue and behavior
- 730d (24m): Major gifts

The `H` parameter is retained for backward compatibility but does not control label horizons.

### Categorical Features
Several features have categorical values that may require encoding:
- `tenure_bucket` (ordinal: "0-3m", "3m-1y", "1y-3y", "3y+")
- `latest_project_category`, `latest_project_grade`, `latest_referral_channel` (nominal)
- `label_next_gift_category`, `label_next_gift_grade_band`, `label_quarter_of_max_future_giving_next_12m` (nominal)

**Encoding impact**: After one-hot encoding categorical features, the final feature space can expand to 200+ columns depending on category cardinality.


## 7. Notes & Best Practices

### Time Scope Summary

- **STATIC**: Does not depend on T
  - ZIP/ACS features
  - Some "once ever" flags (is_major_gift_donor, etc.)
  - First donation seasonality

- **AS_OF_T**: Cumulative through time T
  - Lifetime giving metrics
  - Distance/locality
  - Channel mix (lifetime)
  - Teacher/school concentration
  - Overall seasonality patterns

- **WINDOWED**: Specific lookback relative to T
  - 3m/12m/36m counts & amounts
  - Velocity metrics
  - Email engagement (3m/12m)
  - Site behavior (3m)
  - Share activity (12m)

- **LABEL**: Future horizon [T, T+H)
  - Repeat giving targets
  - Monthly program changes
  - Upgrade/downgrade indicators

### Key Design Decisions

1. **No duplicate concepts**: Single distance metric, single velocity pattern
2. **Consistent windows**: 3m (short), 12m (mid), 36m (long)
3. **Defensive epsilon**: Small values (1e-6) prevent division by zero
4. **Separation of concerns**: Feature engineering ≠ imputation
5. **Time-relative**: All features parameterized by T

### Recommended Workflow

1. **Build features** for each training example at different T values
2. **Finalize features** with consistent imputation strategy
3. **Feature selection** based on importance/correlation
4. **Model training** with cross-validation
5. **Production scoring** with latest T value

### Common Pitfalls to Avoid

- **Leakage**: Don't use features from [T, T+H) for predictions
- **Inconsistent windows**: Always use same T-relative boundaries
- **Missing joins**: Ensure all tables have donor_id
- **Scale mismatch**: Consider feature scaling for some models
- **Category explosion**: Monitor one-hot encoding dimension

### Performance Tips

- For large datasets, consider building features in chunks
- Cache intermediate results (windowed subsets)
- Use categorical dtypes to save memory
- Consider feature hashing for high-cardinality categoricals
- Profile your code to identify bottlenecks

### Extensions

This schema can be extended with:
- Interaction features (e.g., tenure × amount)
- Polynomial features for non-linear patterns
- Embedding features from text (project descriptions)
- Time-series features (rolling statistics)
- Graph features (donor networks)
- External data (economic indicators, events)